In [47]:
import sys

print(sys.executable)

c:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\.venv\Scripts\python.exe


In [48]:
import duckdb
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

In [49]:
con = duckdb.connect()
print("DuckDB connection established.")

DuckDB connection established.


In [50]:
PROJECT_ROOT = Path("..")

DATASET_DIR = PROJECT_ROOT / "dataset"
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

print("Train directory:", TRAIN_DIR.resolve())
print("Test directory:", TEST_DIR.resolve())

Train directory: C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\dataset\train
Test directory: C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\dataset\test


In [51]:
SOURCE1_PATH = (TRAIN_DIR / "train_source1.tsv").resolve()
SOURCE2_PATH = (TRAIN_DIR / "train_source2.tsv").resolve()
SOURCE3_PATH = (TRAIN_DIR / "train_source3.tsv").resolve()

print(SOURCE1_PATH)
print(SOURCE2_PATH)
print(SOURCE3_PATH)

C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\dataset\train\train_source1.tsv
C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\dataset\train\train_source2.tsv
C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\dataset\train\train_source3.tsv


In [52]:
source1_test = con.execute(f"""
    SELECT *
    FROM read_csv_auto(
        '{SOURCE1_PATH.as_posix()}',
        delim='\\t',
        header=true
    )
    LIMIT 10
""").df()

source1_test

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US


In [8]:
source1_count = con.execute(f"""
    SELECT COUNT(*)
    FROM read_csv_auto(
        '{SOURCE1_PATH.as_posix()}',
        delim='\\t',
        header=true
    )
""").fetchone()[0]

print("Source 1 rows:", source1_count)

Source 1 rows: 2206821


In [9]:
def normalize_text(text):
    if text is None or pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Lowercase
    text = text.lower()

    # Keep letters, marks, numbers and whitespace.
    # Replace punctuation/symbols with spaces.
    cleaned = []

    for char in text:
        category = unicodedata.category(char)

        if category[0] in {"L", "M", "N"} or char.isspace():
            cleaned.append(char)
        else:
            cleaned.append(" ")

    text = "".join(cleaned)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [10]:
examples = [
    "ABC Technologies Pvt. Ltd.",
    "ABC TECHNOLOGIES PVT LTD",
    "McDonald's India Pvt. Ltd.",
    "B+ Retail Inc",
    "राम मार्केटिंग प्राइवेट लिमिटेड",
    "LLC Moncada Learning Center"
]

for value in examples:
    print(f"{value}")
    print(f"→ {normalize_text(value)}")
    print()

ABC Technologies Pvt. Ltd.
→ abc technologies pvt ltd

ABC TECHNOLOGIES PVT LTD
→ abc technologies pvt ltd

McDonald's India Pvt. Ltd.
→ mcdonald s india pvt ltd

B+ Retail Inc
→ b retail inc

राम मार्केटिंग प्राइवेट लिमिटेड
→ राम मार्केटिंग प्राइवेट लिमिटेड

LLC Moncada Learning Center
→ llc moncada learning center



In [11]:
sample_query = f"""
    SELECT entity_id, business_name, country
    FROM read_csv_auto(
        '{SOURCE1_PATH.as_posix()}',
        delim='\\t',
        header=true
    )
    USING SAMPLE 20%
    LIMIT 20
"""

sample = con.execute(sample_query).df()

sample["normalized_name"] = sample["business_name"].apply(normalize_text)

sample[["business_name", "normalized_name", "country"]]

,business_name,normalized_name,country
0,Hamilton Processing Associates,hamilton processing associates,US
1,Tejika Entertainment Private Limited,tejika entertainment private limited,India
2,Amick and Martinez Center,amick and martinez center,US
3,Physical Therapy Specialists LLC,physical therapy specialists llc,US
4,Yamuanth International Private Limited,yamuanth international private limited,India
5,Vats Nursing Home,vats nursing home,India
6,Delhi Cold Private Limited,delhi cold private limited,India
7,Solutions Nirmala Tourism Private Limited,solutions nirmala tourism private limited,India
8,Urgent Care Group PLLC,urgent care group pllc,US
9,Eleven Advisors,eleven advisors,India


In [12]:
sample = con.execute(f"""
    SELECT entity_id, business_name, country
    FROM read_csv_auto(
        '{SOURCE1_PATH.as_posix()}',
        delim='\\t',
        header=true
    )
    LIMIT 100
""").df()

sample["normalized_name"] = sample["business_name"].apply(normalize_text)

sample[["business_name", "normalized_name", "country"]]

,business_name,normalized_name,country
0,Orelee's Barbershop,orelee s barbershop,US
1,Prime Money,prime money,US
2,B+ Retail Inc,b retail inc,US
3,Christ Chapel,christ chapel,US
4,Prabhav Business Center,prabhav business center,India
...,...,...,...
95,B+ Biotechnologies LLC,b biotechnologies llc,US
96,"Pocius, Herrera & Lima Connecticut Inc",pocius herrera lima connecticut inc,US
97,Murray & Cruz Inc,murray cruz inc,US
98,Vt Exporters,vt exporters,India


In [13]:
test_query = f"""
SELECT
    business_name,
    country,
    lower(trim(business_name)) AS simple_name
FROM read_csv_auto(
    '{SOURCE2_PATH.as_posix()}',
    delim='\\t',
    header=true
)
LIMIT 20
"""

duckdb_test = con.execute(test_query).df()

duckdb_test

,business_name,country,simple_name
0,राम मार्केटिंग प्राइवेट लिमिटेड,India,राम मार्केटिंग प्राइवेट लिमिटेड
1,-- Holloway Peak Inc Seafood,US,-- holloway peak inc seafood
2,आदित्य प्रॉपर्टीज एलएलपी,India,आदित्य प्रॉपर्टीज एलएलपी
3,Summit Inc,US,summit inc
4,Delta Tetlecommunication Inc,US,delta tetlecommunication inc
5,Lee and Lawson,US,lee and lawson
6,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,India,shivshakti vidyalaya vidyalaya overseas corpor...
7,Shree Infracon Private Ltd,India,shree infracon private ltd
8,Chavira Platinum Chimera LLC,US,chavira platinum chimera llc
9,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,India,foundation excel agency private limited


In [14]:
NAME_NORMALIZATION_SQL = r"""
    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    )
"""

In [15]:
test_query = f"""
SELECT
    business_name,
    country,
    {NAME_NORMALIZATION_SQL} AS normalized_name
FROM read_csv_auto(
    '{SOURCE2_PATH.as_posix()}',
    delim='\\t',
    header=true
)
LIMIT 20
"""

duckdb_test = con.execute(test_query).df()

duckdb_test

,business_name,country,normalized_name
0,राम मार्केटिंग प्राइवेट लिमिटेड,India,राम मार्केटिंग प्राइवेट लिमिटेड
1,-- Holloway Peak Inc Seafood,US,holloway peak inc seafood
2,आदित्य प्रॉपर्टीज एलएलपी,India,आदित्य प्रॉपर्टीज एलएलपी
3,Summit Inc,US,summit inc
4,Delta Tetlecommunication Inc,US,delta tetlecommunication inc
5,Lee and Lawson,US,lee and lawson
6,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,India,shivshakti vidyalaya vidyalaya overseas corpor...
7,Shree Infracon Private Ltd,India,shree infracon private ltd
8,Chavira Platinum Chimera LLC,US,chavira platinum chimera llc
9,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,India,foundation excel agency private limited


In [16]:
source2_name_index_query = f"""
SELECT
    {NAME_NORMALIZATION_SQL} AS normalized_name,
    COUNT(*) AS record_count
FROM read_csv_auto(
    '{SOURCE2_PATH.as_posix()}',
    delim='\\t',
    header=true
)
GROUP BY normalized_name
ORDER BY record_count DESC
LIMIT 20
"""

source2_name_index = con.execute(
    source2_name_index_query
).df()

source2_name_index

,normalized_name,record_count
0,primary care,489
1,physical therapy,481
2,womens health,447
3,urgent care,435
4,behavioral health,435
5,pediatric dental,429
6,internal medicine,426
7,pediatric dentistry,425
8,earnosethroat com,348
9,foot ankle,337


In [17]:
source3_name_index_query = f"""
SELECT
    {NAME_NORMALIZATION_SQL} AS normalized_name,
    COUNT(*) AS record_count
FROM read_csv_auto(
    '{SOURCE3_PATH.as_posix()}',
    delim='\\t',
    header=true
)
GROUP BY normalized_name
ORDER BY record_count DESC
LIMIT 20
"""

source3_name_index = con.execute(
    source3_name_index_query
).df()

source3_name_index

,normalized_name,record_count
0,primary care,491
1,physical therapy,480
2,pediatric dental,473
3,urgent care,450
4,womens health,440
5,pediatric dentistry,432
6,internal medicine,398
7,behavioral health,394
8,foot ankle,322
9,pediatric,321


In [18]:
con.execute(f"""
CREATE OR REPLACE VIEW train_source1 AS
SELECT *
FROM read_csv_auto(
    '{SOURCE1_PATH.as_posix()}',
    delim='\\t',
    header=true
)
""")

con.execute(f"""
CREATE OR REPLACE VIEW train_source2 AS
SELECT *
FROM read_csv_auto(
    '{SOURCE2_PATH.as_posix()}',
    delim='\\t',
    header=true
)
""")

con.execute(f"""
CREATE OR REPLACE VIEW train_source3 AS
SELECT *
FROM read_csv_auto(
    '{SOURCE3_PATH.as_posix()}',
    delim='\\t',
    header=true
)
""")

print("Training source views created.")

Training source views created.


In [19]:
con.execute("""
SELECT COUNT(*) FROM train_source1
""").fetchone()

(2206821,)

In [20]:
con.execute("""
SELECT COUNT(*) FROM train_source2
""").fetchone()

(5034616,)

In [21]:
con.execute("""
SELECT COUNT(*) FROM train_source3
""").fetchone()

(5285603,)

In [22]:
BLOCKING_KEY_SQL = """
    concat(
        country,
        '||',
        regexp_replace(
            regexp_replace(
                lower(business_name),
                '[^\\p{L}\\p{M}\\p{N}\\s]',
                ' ',
                'g'
            ),
            '\\s+',
            ' ',
            'g'
        )
    )
"""

In [23]:
source2_blocking_stats = con.execute(f"""
SELECT
    {BLOCKING_KEY_SQL} AS blocking_key,
    COUNT(*) AS record_count
FROM train_source2
GROUP BY blocking_key
ORDER BY record_count DESC
LIMIT 20
""").df()

source2_blocking_stats

,blocking_key,record_count
0,US||primary care,489
1,US||physical therapy,481
2,US||womens health,447
3,US||urgent care,435
4,US||behavioral health,435
5,US||pediatric dental,429
6,US||internal medicine,426
7,US||pediatric dentistry,425
8,US||earnosethroat com,348
9,US||ear nose throat,337


In [24]:
source3_blocking_stats = con.execute(f"""
SELECT
    {BLOCKING_KEY_SQL} AS blocking_key,
    COUNT(*) AS record_count
FROM train_source3
GROUP BY blocking_key
ORDER BY record_count DESC
LIMIT 20
""").df()

source3_blocking_stats

,blocking_key,record_count
0,US||primary care,491
1,US||physical therapy,480
2,US||pediatric dental,473
3,US||urgent care,450
4,US||womens health,440
5,US||pediatric dentistry,432
6,US||internal medicine,398
7,US||behavioral health,394
8,US||foot ankle,322
9,US||pediatric,321


In [25]:
source2_block_distribution = con.execute(f"""
SELECT
    {BLOCKING_KEY_SQL} AS blocking_key,
    COUNT(*) AS block_size
FROM train_source2
GROUP BY blocking_key
""").df()

source2_block_distribution["block_size"].describe()

count    4.120458e+06
mean     1.221858e+00
std      1.801017e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      4.890000e+02
Name: block_size, dtype: float64

In [26]:
source3_block_distribution = con.execute(f"""
SELECT
    {BLOCKING_KEY_SQL} AS blocking_key,
    COUNT(*) AS block_size
FROM train_source3
GROUP BY blocking_key
""").df()

source3_block_distribution["block_size"].describe()

count    4.389842e+06
mean     1.204053e+00
std      1.748927e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      4.910000e+02
Name: block_size, dtype: float64

In [27]:
con.execute(f"""
CREATE OR REPLACE VIEW source1_keys AS
SELECT
    entity_id,
    {BLOCKING_KEY_SQL} AS blocking_key
FROM train_source1
""")

print("Source 1 normalized-key view created.")

Source 1 normalized-key view created.


In [28]:
con.execute(f"""
CREATE OR REPLACE VIEW source2_keys AS
SELECT
    entity_id,
    {BLOCKING_KEY_SQL} AS blocking_key
FROM train_source2
""")

print("Source 2 normalized-key view created.")

Source 2 normalized-key view created.


In [29]:
con.execute(f"""
CREATE OR REPLACE VIEW source3_keys AS
SELECT
    entity_id,
    {BLOCKING_KEY_SQL} AS blocking_key
FROM train_source3
""")

print("Source 3 normalized-key view created.")

Source 3 normalized-key view created.


In [30]:
con.execute(f"""
CREATE OR REPLACE VIEW ground_truth_matches AS

SELECT
    source1_entity_id,
    matched_entity_id
FROM (
    SELECT
        source1_entity_id,
        unnest(string_split(
            COALESCE(matched_entity_ids, ''),
            ','
        )) AS matched_entity_id
    FROM read_csv_auto(
        '{(TRAIN_DIR / "train_ground_truth.tsv").resolve().as_posix()}',
        delim='\\t',
        header=true
    )
)
WHERE matched_entity_id != ''
""")

print("Ground-truth matches view created.")

Ground-truth matches view created.


In [31]:
con.execute("""
SELECT *
FROM ground_truth_matches
LIMIT 20
""").df()

,source1_entity_id,matched_entity_id
0,S1-965667,S2-681193310
1,S1-965667,S2-743505751
2,S1-965667,S3-775321672
3,S1-965667,S3-11291185
4,S1-965667,S3-860443364
5,S1-55344266,S2-249013014
6,S1-55344266,S2-197070651
7,S1-55344266,S3-478195123
8,S1-55344266,S3-384364074
9,S1-343815751,S2-790675320


In [32]:
s2_recall = con.execute("""
SELECT
    COUNT(*) AS total_true_matches,
    SUM(
        CASE
            WHEN s1.blocking_key = s2.blocking_key
            THEN 1
            ELSE 0
        END
    ) AS matches_captured
FROM ground_truth_matches gt
JOIN source1_keys s1
    ON gt.source1_entity_id = s1.entity_id
JOIN source2_keys s2
    ON gt.matched_entity_id = s2.entity_id
WHERE gt.matched_entity_id LIKE 'S2-%'
""").df()

s2_recall

,total_true_matches,matches_captured
0,3693619,665725.0


In [33]:
s2_total = int(s2_recall.loc[0, "total_true_matches"])
s2_captured = int(s2_recall.loc[0, "matches_captured"])

s2_candidate_recall = s2_captured / s2_total

print("S2 true matches:", s2_total)
print("S2 matches captured:", s2_captured)
print(f"S2 candidate recall: {s2_candidate_recall:.4%}")

S2 true matches: 3693619
S2 matches captured: 665725
S2 candidate recall: 18.0237%


In [34]:
s3_recall = con.execute("""
SELECT
    COUNT(*) AS total_true_matches,
    SUM(
        CASE
            WHEN s1.blocking_key = s3.blocking_key
            THEN 1
            ELSE 0
        END
    ) AS matches_captured
FROM ground_truth_matches gt
JOIN source1_keys s1
    ON gt.source1_entity_id = s1.entity_id
JOIN source3_keys s3
    ON gt.matched_entity_id = s3.entity_id
WHERE gt.matched_entity_id LIKE 'S3-%'
""").df()

s3_recall

,total_true_matches,matches_captured
0,3944746,736167.0


In [35]:
s3_total = int(s3_recall.loc[0, "total_true_matches"])
s3_captured = int(s3_recall.loc[0, "matches_captured"])

s3_candidate_recall = s3_captured / s3_total

print("S3 true matches:", s3_total)
print("S3 matches captured:", s3_captured)
print(f"S3 candidate recall: {s3_candidate_recall:.4%}")

S3 true matches: 3944746
S3 matches captured: 736167
S3 candidate recall: 18.6620%


In [36]:
NAME_ONLY_SQL = r"""
    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    )
"""

In [37]:
source2_token_stats = con.execute(f"""
SELECT
    token,
    COUNT(*) AS record_count
FROM train_source2,
LATERAL unnest(
    string_split(
        trim({NAME_ONLY_SQL}),
        ' '
    )
) AS t(token)
WHERE token != ''
GROUP BY token
ORDER BY record_count DESC
LIMIT 30
""").df()

source2_token_stats

,token,record_count
0,llc,532379
1,limited,532213
2,private,532072
3,inc,403240
4,ltd,401133
5,com,202135
6,center,196536
7,pvt,190499
8,लिमिटेड,190091
9,l,178692


In [38]:
source3_token_stats = con.execute(f"""
SELECT
    token,
    COUNT(*) AS record_count
FROM train_source3,
LATERAL unnest(
    string_split(
        trim({NAME_ONLY_SQL}),
        ' '
    )
) AS t(token)
WHERE token != ''
GROUP BY token
ORDER BY record_count DESC
LIMIT 30
""").df()

source3_token_stats

,token,record_count
0,limited,677666
1,private,644512
2,llc,575634
3,ltd,430703
4,inc,422989
5,center,223999
6,pvt,221763
7,com,211884
8,l,169945
9,services,168970


In [39]:
con.execute(f"""
CREATE OR REPLACE VIEW source2_token_frequency AS
SELECT
    token,
    COUNT(*) AS record_count
FROM train_source2,
LATERAL unnest(
    string_split(
        trim({NAME_ONLY_SQL}),
        ' '
    )
) AS t(token)
WHERE token != ''
GROUP BY token
""")

print("Source 2 token-frequency view created.")

Source 2 token-frequency view created.


In [40]:
con.execute(f"""
CREATE OR REPLACE VIEW source3_token_frequency AS
SELECT
    token,
    COUNT(*) AS record_count
FROM train_source3,
LATERAL unnest(
    string_split(
        trim({NAME_ONLY_SQL}),
        ' '
    )
) AS t(token)
WHERE token != ''
GROUP BY token
""")

print("Source 3 token-frequency view created.")

Source 3 token-frequency view created.


In [41]:
con.execute("""
SELECT *
FROM source2_token_frequency
WHERE record_count <= 10
ORDER BY record_count DESC
LIMIT 30
""").df()

,token,record_count
0,kolam,10
1,brue,10
2,xsr,10
3,moreva,10
4,3405,10
5,mcgillicuddy,10
6,vco,10
7,eeco,10
8,chiranjivi,10
9,glidewell,10


In [42]:
con.execute("""
SELECT *
FROM source3_token_frequency
WHERE record_count <= 10
ORDER BY record_count DESC
LIMIT 30
""").df()

,token,record_count
0,molter,10
1,xwj,10
2,djb,10
3,weyer,10
4,pinera,10
5,crus,10
6,flavin,10
7,sowden,10
8,cárpenter,10
9,elemental,10


In [43]:
thresholds = [100, 500, 1000, 5000, 10000]

s2_token_recall_results = []

for threshold in thresholds:

    result = con.execute(f"""
    WITH
    informative_tokens AS (
        SELECT token
        FROM source2_token_frequency
        WHERE record_count <= {threshold}
    ),

    true_pairs AS (
        SELECT
            source1_entity_id,
            matched_entity_id
        FROM ground_truth_matches
        WHERE matched_entity_id LIKE 'S2-%'
    ),

    pair_tokens AS (
        SELECT
            tp.source1_entity_id,
            tp.matched_entity_id,

            regexp_split_to_array(
                trim(
                    regexp_replace(
                        regexp_replace(
                            lower(s1.business_name),
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}\\s]',
                            ' ',
                            'g'
                        ),
                        '\\s+',
                        ' ',
                        'g'
                    )
                ),
                '\\s+'
            ) AS s1_tokens,

            regexp_split_to_array(
                trim(
                    regexp_replace(
                        regexp_replace(
                            lower(s2.business_name),
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}\\s]',
                            ' ',
                            'g'
                        ),
                        '\\s+',
                        ' ',
                        'g'
                    )
                ),
                '\\s+'
            ) AS s2_tokens

        FROM true_pairs tp

        JOIN train_source1 s1
            ON tp.source1_entity_id = s1.entity_id

        JOIN train_source2 s2
            ON tp.matched_entity_id = s2.entity_id
    ),

    matches AS (
        SELECT
            pt.*,

            EXISTS (
                SELECT 1
                FROM unnest(pt.s1_tokens) AS a(token)
                JOIN informative_tokens it
                    ON a.token = it.token
                WHERE list_contains(pt.s2_tokens, a.token)
            ) AS captured

        FROM pair_tokens pt
    )

    SELECT
        COUNT(*) AS total_matches,
        SUM(CASE WHEN captured THEN 1 ELSE 0 END) AS captured_matches
    FROM matches
    """).fetchone()

    total = result[0]
    captured = result[1]

    s2_token_recall_results.append({
        "threshold": threshold,
        "total_matches": total,
        "captured": captured,
        "recall": captured / total
    })

pd.DataFrame(s2_token_recall_results)

,threshold,total_matches,captured,recall
0,100,3693619,1057408,0.286280
1,500,3693619,1570068,0.425076
2,1000,3693619,1860223,0.503632
3,5000,3693619,2429296,0.657701
4,10000,3693619,2633939,0.713105


In [44]:
thresholds = [100, 500, 1000, 5000, 10000]

s3_token_recall_results = []

for threshold in thresholds:

    result = con.execute(f"""
    WITH
    informative_tokens AS (
        SELECT token
        FROM source3_token_frequency
        WHERE record_count <= {threshold}
    ),

    true_pairs AS (
        SELECT
            source1_entity_id,
            matched_entity_id
        FROM ground_truth_matches
        WHERE matched_entity_id LIKE 'S3-%'
    ),

    pair_tokens AS (
        SELECT
            tp.source1_entity_id,
            tp.matched_entity_id,

            regexp_split_to_array(
                trim(
                    regexp_replace(
                        regexp_replace(
                            lower(s1.business_name),
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}\\s]',
                            ' ',
                            'g'
                        ),
                        '\\s+',
                        ' ',
                        'g'
                    )
                ),
                '\\s+'
            ) AS s1_tokens,

            regexp_split_to_array(
                trim(
                    regexp_replace(
                        regexp_replace(
                            lower(s3.business_name),
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}\\s]',
                            ' ',
                            'g'
                        ),
                        '\\s+',
                        ' ',
                        'g'
                    )
                ),
                '\\s+'
            ) AS s3_tokens

        FROM true_pairs tp

        JOIN train_source1 s1
            ON tp.source1_entity_id = s1.entity_id

        JOIN train_source3 s3
            ON tp.matched_entity_id = s3.entity_id
    ),

    matches AS (
        SELECT
            pt.*,

            EXISTS (
                SELECT 1
                FROM unnest(pt.s1_tokens) AS a(token)
                JOIN informative_tokens it
                    ON a.token = it.token
                WHERE list_contains(pt.s3_tokens, a.token)
            ) AS captured

        FROM pair_tokens pt
    )

    SELECT
        COUNT(*) AS total_matches,
        SUM(CASE WHEN captured THEN 1 ELSE 0 END) AS captured_matches
    FROM matches
    """).fetchone()

    total = result[0]
    captured = result[1]

    s3_token_recall_results.append({
        "threshold": threshold,
        "total_matches": total,
        "captured": captured,
        "recall": captured / total
    })

pd.DataFrame(s3_token_recall_results)

,threshold,total_matches,captured,recall
0,100,3944746,1097360,0.278183
1,500,3944746,1657143,0.420089
2,1000,3944746,1949161,0.494116
3,5000,3944746,2448945,0.620812
4,10000,3944746,2841473,0.720318


In [45]:
combined_s2_recall = con.execute("""
WITH
informative_tokens AS (
    SELECT token
    FROM source2_token_frequency
    WHERE record_count <= 10000
),

true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S2-%'
),

pair_data AS (
    SELECT
        tp.source1_entity_id,
        tp.matched_entity_id,

        s1.blocking_key AS s1_key,
        s2.blocking_key AS s2_key,

        regexp_split_to_array(
            trim(
                regexp_replace(
                    regexp_replace(
                        lower(s1.business_name),
                        '[^\p{L}\p{M}\p{N}\s]',
                        ' ',
                        'g'
                    ),
                    '\s+',
                    ' ',
                    'g'
                )
            ),
            '\s+'
        ) AS s1_tokens,

        regexp_split_to_array(
            trim(
                regexp_replace(
                    regexp_replace(
                        lower(s2.business_name),
                        '[^\p{L}\p{M}\p{N}\s]',
                        ' ',
                        'g'
                    ),
                    '\s+',
                    ' ',
                    'g'
                )
            ),
            '\s+'
        ) AS s2_tokens

    FROM true_pairs tp

    JOIN train_source1 s1
        ON tp.source1_entity_id = s1.entity_id

    JOIN train_source2 s2
        ON tp.matched_entity_id = s2.entity_id
),

captured AS (
    SELECT
        *,
        (
            s1_key = s2_key
        )
        OR
        EXISTS (
            SELECT 1
            FROM unnest(s1_tokens) AS a(token)
            JOIN informative_tokens it
                ON a.token = it.token
            WHERE list_contains(s2_tokens, a.token)
        ) AS is_captured

    FROM pair_data
)

SELECT
    COUNT(*) AS total_matches,
    SUM(CASE WHEN is_captured THEN 1 ELSE 0 END) AS captured_matches
FROM captured
""").df()

combined_s2_recall

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\yugan\AppData\Local\Temp\ipykernel_23036\2427443929.py:1: SyntaxWarning: invalid escape sequence '\p'
  combined_s2_recall = con.execute("""


BinderException: Binder Error: Values list "s1" does not have a column named "blocking_key"

LINE 22:         s1.blocking_key AS s1_key,
                 ^

In [ ]:
combined_s2_recall = con.execute(r"""
WITH
informative_tokens AS (
    SELECT token
    FROM source2_token_frequency
    WHERE record_count <= 10000
),

true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S2-%'
),

pair_data AS (
    SELECT
        tp.source1_entity_id,
        tp.matched_entity_id,

        s1.blocking_key AS s1_key,
        s2.blocking_key AS s2_key,

        regexp_split_to_array(
            trim(
                regexp_replace(
                    regexp_replace(
                        lower(s1_raw.business_name),
                        '[^\p{L}\p{M}\p{N}\s]',
                        ' ',
                        'g'
                    ),
                    '\s+',
                    ' ',
                    'g'
                )
            ),
            '\s+'
        ) AS s1_tokens,

        regexp_split_to_array(
            trim(
                regexp_replace(
                    regexp_replace(
                        lower(s2_raw.business_name),
                        '[^\p{L}\p{M}\p{N}\s]',
                        ' ',
                        'g'
                    ),
                    '\s+',
                    ' ',
                    'g'
                )
            ),
            '\s+'
        ) AS s2_tokens

    FROM true_pairs tp

    JOIN source1_keys s1
        ON tp.source1_entity_id = s1.entity_id

    JOIN source2_keys s2
        ON tp.matched_entity_id = s2.entity_id

    JOIN train_source1 s1_raw
        ON tp.source1_entity_id = s1_raw.entity_id

    JOIN train_source2 s2_raw
        ON tp.matched_entity_id = s2_raw.entity_id
),

captured AS (
    SELECT
        *,
        (
            s1_key = s2_key
        )
        OR
        EXISTS (
            SELECT 1
            FROM unnest(s1_tokens) AS a(token)
            JOIN informative_tokens it
                ON a.token = it.token
            WHERE list_contains(s2_tokens, a.token)
        ) AS is_captured

    FROM pair_data
)

SELECT
    COUNT(*) AS total_matches,
    SUM(
        CASE
            WHEN is_captured THEN 1
            ELSE 0
        END
    ) AS captured_matches
FROM captured
""").df()

combined_s2_recall

,total_matches,captured_matches
0,3693619,2704016.0


In [ ]:
s2_combined_total = int(
    combined_s2_recall.loc[0, "total_matches"]
)

s2_combined_captured = int(
    combined_s2_recall.loc[0, "captured_matches"]
)

print("S2 true matches:", s2_combined_total)
print("S2 captured:", s2_combined_captured)
print(
    f"S2 combined recall: "
    f"{s2_combined_captured / s2_combined_total:.4%}"
)

S2 true matches: 3693619
S2 captured: 2704016
S2 combined recall: 73.2078%


In [ ]:
combined_s3_recall = con.execute(r"""
WITH
informative_tokens AS (
    SELECT token
    FROM source3_token_frequency
    WHERE record_count <= 10000
),

true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S3-%'
),

pair_data AS (
    SELECT
        tp.source1_entity_id,
        tp.matched_entity_id,

        s1.blocking_key AS s1_key,
        s3.blocking_key AS s3_key,

        regexp_split_to_array(
            trim(
                regexp_replace(
                    regexp_replace(
                        lower(s1_raw.business_name),
                        '[^\p{L}\p{M}\p{N}\s]',
                        ' ',
                        'g'
                    ),
                    '\s+',
                    ' ',
                    'g'
                )
            ),
            '\s+'
        ) AS s1_tokens,

        regexp_split_to_array(
            trim(
                regexp_replace(
                    regexp_replace(
                        lower(s3_raw.business_name),
                        '[^\p{L}\p{M}\p{N}\s]',
                        ' ',
                        'g'
                    ),
                    '\s+',
                    ' ',
                    'g'
                )
            ),
            '\s+'
        ) AS s3_tokens

    FROM true_pairs tp

    JOIN source1_keys s1
        ON tp.source1_entity_id = s1.entity_id

    JOIN source3_keys s3
        ON tp.matched_entity_id = s3.entity_id

    JOIN train_source1 s1_raw
        ON tp.source1_entity_id = s1_raw.entity_id

    JOIN train_source3 s3_raw
        ON tp.matched_entity_id = s3_raw.entity_id
),

captured AS (
    SELECT
        *,
        (
            s1_key = s3_key
        )
        OR
        EXISTS (
            SELECT 1
            FROM unnest(s1_tokens) AS a(token)
            JOIN informative_tokens it
                ON a.token = it.token
            WHERE list_contains(s3_tokens, a.token)
        ) AS is_captured

    FROM pair_data
)

SELECT
    COUNT(*) AS total_matches,
    SUM(
        CASE
            WHEN is_captured THEN 1
            ELSE 0
        END
    ) AS captured_matches
FROM captured
""").df()

combined_s3_recall

,total_matches,captured_matches
0,3944746,2933335.0


In [ ]:
s3_combined_total = int(
    combined_s3_recall.loc[0, "total_matches"]
)

s3_combined_captured = int(
    combined_s3_recall.loc[0, "captured_matches"]
)

print("S3 true matches:", s3_combined_total)
print("S3 captured:", s3_combined_captured)

print(
    "S3 combined recall:",
    f"{s3_combined_captured / s3_combined_total:.4%}"
)

S3 true matches: 3944746
S3 captured: 2933335
S3 combined recall: 74.3606%


In [ ]:
ADDRESS_NORMALIZATION_SQL = r"""
    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    )
"""

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW source2_address_token_frequency AS
SELECT
    token,
    COUNT(*) AS record_count
FROM train_source2,
LATERAL unnest(
    string_split(
        trim({ADDRESS_NORMALIZATION_SQL}),
        ' '
    )
) AS t(token)
WHERE token != ''
GROUP BY token
""")

print("Source 2 address-token frequency view created.")

Source 2 address-token frequency view created.


In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW source3_address_token_frequency AS
SELECT
    token,
    COUNT(*) AS record_count
FROM train_source3,
LATERAL unnest(
    string_split(
        trim({ADDRESS_NORMALIZATION_SQL}),
        ' '
    )
) AS t(token)
WHERE token != ''
GROUP BY token
""")

print("Source 3 address-token frequency view created.")

Source 3 address-token frequency view created.


In [ ]:
con.execute("""
SELECT *
FROM source2_address_token_frequency
ORDER BY record_count DESC
LIMIT 30
""").df()

,token,record_count
0,no,1117851
1,road,680160
2,delhi,491311
3,street,338282
4,st,334979
5,rd,331804
6,floor,320613
7,maharashtra,320257
8,1,303779
9,nagar,303686


In [ ]:
con.execute("""
SELECT *
FROM source3_address_token_frequency
ORDER BY record_count DESC
LIMIT 30
""").df()

,token,record_count
0,no,1073672
1,road,627839
2,new,479832
3,delhi,447113
4,street,357908
5,city,356110
6,st,332829
7,north,332787
8,1,330088
9,rd,326949


In [ ]:
con.execute("""
SELECT *
FROM source2_address_token_frequency
WHERE record_count <= 10
ORDER BY record_count DESC
LIMIT 30
""").df()

,token,record_count
0,15rohini,10
1,bhonsale,10
2,koehne,10
3,14548,10
4,klahanie,10
5,plummers,10
6,claremoore,10
7,3180a,10
8,b405,10
9,branchview,10


In [ ]:
con.execute("""
SELECT *
FROM source3_address_token_frequency
WHERE record_count <= 10
ORDER BY record_count DESC
LIMIT 30
""").df()

,token,record_count
0,groumd,10
1,bangesh,10
2,vart,10
3,suvarnaratna,10
4,jefferis,10
5,006705,10
6,mayyil,10
7,31450,10
8,qazipur,10
9,1100c,10


In [ ]:
thresholds = [100, 500, 1000, 5000, 10000]

s2_address_recall_results = []

for threshold in thresholds:

    result = con.execute(f"""
    WITH
    informative_tokens AS (
        SELECT token
        FROM source2_address_token_frequency
        WHERE record_count <= {threshold}
    ),

    true_pairs AS (
        SELECT
            source1_entity_id,
            matched_entity_id
        FROM ground_truth_matches
        WHERE matched_entity_id LIKE 'S2-%'
    ),

    pair_data AS (
        SELECT
            tp.source1_entity_id,
            tp.matched_entity_id,

            regexp_split_to_array(
                trim(
                    regexp_replace(
                        regexp_replace(
                            lower(s1.business_address),
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}\\s]',
                            ' ',
                            'g'
                        ),
                        '\\s+',
                        ' ',
                        'g'
                    )
                ),
                '\\s+'
            ) AS s1_tokens,

            regexp_split_to_array(
                trim(
                    regexp_replace(
                        regexp_replace(
                            lower(s2.business_address),
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}\\s]',
                            ' ',
                            'g'
                        ),
                        '\\s+',
                        ' ',
                        'g'
                    )
                ),
                '\\s+'
            ) AS s2_tokens

        FROM true_pairs tp

        JOIN train_source1 s1
            ON tp.source1_entity_id = s1.entity_id

        JOIN train_source2 s2
            ON tp.matched_entity_id = s2.entity_id

        WHERE
            s1.business_address IS NOT NULL
            AND s2.business_address IS NOT NULL
    ),

    matches AS (
        SELECT
            *,
            EXISTS (
                SELECT 1
                FROM unnest(s1_tokens) AS a(token)
                JOIN informative_tokens it
                    ON a.token = it.token
                WHERE list_contains(s2_tokens, a.token)
            ) AS captured
        FROM pair_data
    )

    SELECT
        COUNT(*) AS total_pairs_with_addresses,
        SUM(
            CASE
                WHEN captured THEN 1
                ELSE 0
            END
        ) AS captured_matches
    FROM matches
    """).fetchone()

    total = result[0]
    captured = result[1]

    s2_address_recall_results.append({
        "threshold": threshold,
        "total_pairs_with_addresses": total,
        "captured": captured,
        "recall": captured / total
    })

pd.DataFrame(s2_address_recall_results)

,threshold,total_pairs_with_addresses,captured,recall
0,100,3528435,1547131,0.438475
1,500,3528435,2531547,0.717470
2,1000,3528435,2905728,0.823518
3,5000,3528435,3388397,0.960312
4,10000,3528435,3462385,0.981281


In [ ]:
thresholds = [100, 500, 1000, 5000, 10000]

s3_address_recall_results = []

for threshold in thresholds:

    result = con.execute(f"""
    WITH
    informative_tokens AS (
        SELECT token
        FROM source3_address_token_frequency
        WHERE record_count <= {threshold}
    ),

    true_pairs AS (
        SELECT
            source1_entity_id,
            matched_entity_id
        FROM ground_truth_matches
        WHERE matched_entity_id LIKE 'S3-%'
    ),

    pair_data AS (
        SELECT
            tp.source1_entity_id,
            tp.matched_entity_id,

            regexp_split_to_array(
                trim(
                    regexp_replace(
                        regexp_replace(
                            lower(s1.business_address),
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}\\s]',
                            ' ',
                            'g'
                        ),
                        '\\s+',
                        ' ',
                        'g'
                    )
                ),
                '\\s+'
            ) AS s1_tokens,

            regexp_split_to_array(
                trim(
                    regexp_replace(
                        regexp_replace(
                            lower(s3.business_address),
                            '[^\\p{{L}}\\p{{M}}\\p{{N}}\\s]',
                            ' ',
                            'g'
                        ),
                        '\\s+',
                        ' ',
                        'g'
                    )
                ),
                '\\s+'
            ) AS s3_tokens

        FROM true_pairs tp

        JOIN train_source1 s1
            ON tp.source1_entity_id = s1.entity_id

        JOIN train_source3 s3
            ON tp.matched_entity_id = s3.entity_id

        WHERE
            s1.business_address IS NOT NULL
            AND s3.business_address IS NOT NULL
    ),

    matches AS (
        SELECT
            *,
            EXISTS (
                SELECT 1
                FROM unnest(s1_tokens) AS a(token)
                JOIN informative_tokens it
                    ON a.token = it.token
                WHERE list_contains(s3_tokens, a.token)
            ) AS captured
        FROM pair_data
    )

    SELECT
        COUNT(*) AS total_pairs_with_addresses,
        SUM(
            CASE
                WHEN captured THEN 1
                ELSE 0
            END
        ) AS captured_matches
    FROM matches
    """).fetchone()

    total = result[0]
    captured = result[1]

    s3_address_recall_results.append({
        "threshold": threshold,
        "total_pairs_with_addresses": total,
        "captured": captured,
        "recall": captured / total
    })

pd.DataFrame(s3_address_recall_results)

,threshold,total_pairs_with_addresses,captured,recall
0,100,3772912,1552486,0.411482
1,500,3772912,2567034,0.680385
2,1000,3772912,2961593,0.784962
3,5000,3772912,3524481,0.934154
4,10000,3772912,3628624,0.961757


In [ ]:
con.execute(r"""
CREATE OR REPLACE TABLE s1_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM train_source1
""")

print("S1 normalized table created.")

S1 normalized table created.


In [ ]:
con.execute(r"""
CREATE OR REPLACE TABLE s2_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM train_source2
""")

print("S2 normalized table created.")

S2 normalized table created.


In [ ]:
con.execute(r"""
CREATE OR REPLACE TABLE s3_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM train_source3
""")

print("S3 normalized table created.")

S3 normalized table created.


In [ ]:
con.execute("""
SELECT
    entity_id,
    business_name,
    normalized_name,
    business_address,
    normalized_address,
    country
FROM s2_normalized
LIMIT 10
""").df()

,entity_id,business_name,normalized_name,business_address,normalized_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",kh no 570 13 new delhi west delhi delhi,India
1,S2-764573417,-- Holloway Peak Inc Seafood,holloway peak inc seafood,"105 ELM ST, MORGANTON, NC",105 elm st morganton nc,US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",g 3 571 gulmohar colony bhopal madhya pradesh,India
3,S2-163963287,Summit Inc,summit inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",greensboro nc 19 1 2 stardust trail,US
4,S2-49942811,Delta Tetlecommunication Inc,delta tetlecommunication inc,"914 PIERPONT AVE, CLEVELAND, OH",914 pierpont ave cleveland oh,US
5,S2-138046867,Lee and Lawson,lee and lawson,"1702 Pine Avenue, CITY OF MENOMONIE, WI",1702 pine avenue city of menomonie wi,US
6,S2-584977605,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,shivshakti vidyalaya vidyalaya overseas corpor...,"H.NO 204 C ROAD HOSHIARPUR, PUNJAB, Punjab",h no 204 c road hoshiarpur punjab punjab,India
7,S2-277444929,Shree Infracon Private Ltd,shree infracon private ltd,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",63 2275 7 alhind tower first floor jafferkhan ...,India
8,S2-721031885,Chavira Platinum Chimera LLC,chavira platinum chimera llc,"282 SAXONY DRIVE, FTT MITCHELL, KY",282 saxony drive ftt mitchell ky,US
9,S2-508602797,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,foundation excel agency private limited,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",hn 753 e 1 bharat nagar 104 1 1 erandwane maha...,India


In [ ]:
con.execute(r"""
CREATE OR REPLACE TABLE s2_name_token_index AS
SELECT
    entity_id,
    token
FROM s2_normalized,
LATERAL unnest(
    string_split(normalized_name, ' ')
) AS t(token)
WHERE
    token != ''
    AND token IN (
        SELECT token
        FROM source2_token_frequency
        WHERE record_count <= 10000
    )
""")

print("S2 name-token index created.")

S2 name-token index created.


In [ ]:
con.execute(r"""
CREATE OR REPLACE TABLE s2_address_token_index AS
SELECT
    entity_id,
    token
FROM s2_normalized,
LATERAL unnest(
    string_split(normalized_address, ' ')
) AS t(token)
WHERE
    token != ''
    AND token IN (
        SELECT token
        FROM source2_address_token_frequency
        WHERE record_count <= 10000
    )
""")

print("S2 address-token index created.")

S2 address-token index created.


In [ ]:
print(
    "S2 name index rows:",
    con.execute(
        "SELECT COUNT(*) FROM s2_name_token_index"
    ).fetchone()[0]
)

print(
    "S2 address index rows:",
    con.execute(
        "SELECT COUNT(*) FROM s2_address_token_index"
    ).fetchone()[0]
)

S2 name index rows: 8526025
S2 address index rows: 16688206


In [ ]:
con.execute("""
SELECT *
FROM s2_name_token_index
LIMIT 20
""").df()

,entity_id,token
0,S2-175803113,charities
1,S2-109416570,sab
2,S2-237306523,cóuncil
3,S2-187281716,68613
4,S2-619171364,epr
5,S2-266218107,ássociates
6,S2-229249369,painting
7,S2-497448898,technologiescore
8,S2-540363118,1imited
9,S2-793862769,council


In [ ]:
con.execute("""
SELECT *
FROM s2_address_token_index
LIMIT 20
""").df()

,entity_id,token
0,S2-62166559,baghpat
1,S2-185953974,calicut
2,S2-280643154,ఆంధ్రప్రదేశ్
3,S2-347733054,plymouth
4,S2-430714221,734
5,S2-331081972,pulivendula
6,S2-648155205,payne
7,S2-802104902,varanasi
8,S2-608394840,varanasi
9,S2-84272413,tallassee


In [ ]:
con.execute("""
SELECT
    COUNT(*) AS index_rows,
    COUNT(DISTINCT entity_id) AS entities,
    COUNT(DISTINCT token) AS unique_tokens
FROM s2_name_token_index
""").df()

,index_rows,entities,unique_tokens
0,8526025,4657639,810784


In [ ]:
con.execute("""
SELECT
    token,
    COUNT(*) AS entity_count
FROM s2_name_token_index
GROUP BY token
ORDER BY entity_count DESC
LIMIT 30
""").df()

,token,entity_count
0,ankle,9997
1,greater,9868
2,builders,9718
3,investment,9716
4,westgate,9638
5,projects,9611
6,midtown,9609
7,realty,9597
8,southside,9556
9,riverside,9523


In [ ]:
con.execute("""
SELECT
    COUNT(*) AS index_rows,
    COUNT(DISTINCT entity_id) AS entities,
    COUNT(DISTINCT token) AS unique_tokens
FROM s2_address_token_index
""").df()

,index_rows,entities,unique_tokens
0,16688206,4824103,607462


In [ ]:
con.execute("""
SELECT
    token,
    COUNT(*) AS entity_count
FROM s2_address_token_index
GROUP BY token
ORDER BY entity_count DESC
LIMIT 30
""").df()

,token,entity_count
0,82,10000
1,lincoln,9988
2,401,9959
3,ashok,9945
4,little,9926
5,op,9925
6,buffalo,9914
7,hotel,9914
8,203,9905
9,71,9903


In [ ]:
con.execute(r"""
CREATE OR REPLACE TABLE s1_name_token_index AS
SELECT
    entity_id,
    token
FROM s1_normalized,
LATERAL unnest(
    string_split(normalized_name, ' ')
) AS t(token)
WHERE token != ''
""")

print("S1 name-token index created.")

S1 name-token index created.


In [ ]:
con.execute(r"""
CREATE OR REPLACE TABLE s1_address_token_index AS
SELECT
    entity_id,
    token
FROM s1_normalized,
LATERAL unnest(
    string_split(normalized_address, ' ')
) AS t(token)
WHERE token != ''
""")

print("S1 address-token index created.")

S1 address-token index created.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s1_name_token_frequency AS
SELECT
    token,
    COUNT(*) AS record_count
FROM s1_name_token_index
GROUP BY token
""")

con.execute("""
CREATE OR REPLACE TABLE s1_address_token_frequency AS
SELECT
    token,
    COUNT(*) AS record_count
FROM s1_address_token_index
GROUP BY token
""")

print("S1 token frequency tables created.")

S1 token frequency tables created.


In [ ]:
con.execute("""
SELECT
    token,
    record_count
FROM s1_name_token_frequency
ORDER BY record_count DESC
LIMIT 20
""").df()

,token,record_count
0,limited,522340
1,private,432503
2,llc,355736
3,inc,238309
4,ltd,148599
5,pvt,121462
6,india,61241
7,and,57296
8,s,55791
9,c,54031


In [ ]:
con.execute("""
SELECT
    token,
    record_count
FROM s1_address_token_frequency
ORDER BY record_count DESC
LIMIT 20
""").df()

,token,record_count
0,road,467379
1,no,445587
2,delhi,322098
3,street,309808
4,drive,221567
5,unit,208326
6,maharashtra,192382
7,avenue,189840
8,nagar,166622
9,floor,163098


In [ ]:
con.execute("""
SELECT
    token,
    record_count
FROM s1_name_token_frequency
WHERE record_count <= 10
ORDER BY record_count DESC
LIMIT 30
""").df()

,token,record_count
0,1211,10
1,ratnamani,10
2,tell,10
3,saheli,10
4,brijdhara,10
5,hamsafar,10
6,madani,10
7,mccorvey,10
8,gunnjj,10
9,ngk,10


In [ ]:
con.execute("""
SELECT
    token,
    record_count
FROM s1_address_token_frequency
WHERE record_count <= 10
ORDER BY record_count DESC
LIMIT 30
""").df()

,token,record_count
0,16506,10
1,chamisa,10
2,masuri,10
3,19021,10
4,limber,10
5,chambersa,10
6,surobhi,10
7,21510,10
8,lowes,10
9,18840,10


In [ ]:
s2_country_check = con.execute("""
SELECT
    COUNT(*) AS total_true_matches,
    SUM(
        CASE
            WHEN s1.country = s2.country THEN 1
            ELSE 0
        END
    ) AS same_country
FROM ground_truth_matches gt

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_normalized s2
    ON gt.matched_entity_id = s2.entity_id

WHERE gt.matched_entity_id LIKE 'S2-%'
""").df()

s2_country_check

,total_true_matches,same_country
0,3693619,3693619.0


In [ ]:
s3_country_check = con.execute("""
SELECT
    COUNT(*) AS total_true_matches,
    SUM(
        CASE
            WHEN s1.country = s3.country THEN 1
            ELSE 0
        END
    ) AS same_country
FROM ground_truth_matches gt

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s3_normalized s3
    ON gt.matched_entity_id = s3.entity_id

WHERE gt.matched_entity_id LIKE 'S3-%'
""").df()

s3_country_check

,total_true_matches,same_country
0,3944746,3944746.0


In [ ]:
print(
    "S2 same-country rate:",
    s2_country_check.loc[0, "same_country"]
    / s2_country_check.loc[0, "total_true_matches"]
)

print(
    "S3 same-country rate:",
    s3_country_check.loc[0, "same_country"]
    / s3_country_check.loc[0, "total_true_matches"]
)

S2 same-country rate: 1.0
S3 same-country rate: 1.0


In [ ]:
exact_name_s2 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs
FROM s1_normalized s1
JOIN s2_normalized s2
    ON s1.country = s2.country
   AND s1.normalized_name = s2.normalized_name
WHERE s1.normalized_name != ''
  AND s2.normalized_name != ''
""").fetchone()[0]

print("S1-S2 exact-name candidates:", exact_name_s2)
print(
    "Average candidates per S1:",
    exact_name_s2 / 2206821
)

S1-S2 exact-name candidates: 8608506
Average candidates per S1: 3.9008628248507695


In [ ]:
exact_name_s3 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs
FROM s1_normalized s1
JOIN s3_normalized s3
    ON s1.country = s3.country
   AND s1.normalized_name = s3.normalized_name
WHERE s1.normalized_name != ''
  AND s3.normalized_name != ''
""").fetchone()[0]

print("S1-S3 exact-name candidates:", exact_name_s3)
print(
    "Average candidates per S1:",
    exact_name_s3 / 2206821
)

S1-S3 exact-name candidates: 9473160
Average candidates per S1: 4.292672581962923


In [ ]:
name_token_s2 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs
FROM s1_normalized s1
JOIN s1_name_token_index s1t
    ON s1.entity_id = s1t.entity_id
JOIN s2_name_token_index s2t
    ON s1t.token = s2t.token
JOIN s2_normalized s2
    ON s2.entity_id = s2t.entity_id
WHERE
    s1.country = s2.country
    AND s1t.token IN (
        SELECT token
        FROM s1_name_token_frequency
        WHERE record_count <= 10000
    )
""").fetchone()[0]

print("S1-S2 name-token candidate pairs:", name_token_s2)
print(
    "Average candidates per S1:",
    name_token_s2 / 2206821
)

S1-S2 name-token candidate pairs: 6142417162
Average candidates per S1: 2783.3780637396508


In [ ]:
name_token_s3 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs
FROM s1_normalized s1
JOIN s1_name_token_index s1t
    ON s1.entity_id = s1t.entity_id
JOIN s3_name_token_index s3t
    ON s1t.token = s3t.token
JOIN s3_normalized s3
    ON s3.entity_id = s3t.entity_id
WHERE
    s1.country = s3.country
    AND s1t.token IN (
        SELECT token
        FROM s1_name_token_frequency
        WHERE record_count <= 10000
    )
""").fetchone()[0]

print("S1-S3 name-token candidate pairs:", name_token_s3)
print(
    "Average candidates per S1:",
    name_token_s3 / 2206821
)

CatalogException: Catalog Error: Table with name s3_name_token_index does not exist!
Did you mean "s1_name_token_index"?

LINE 7: JOIN s3_name_token_index s3t
             ^

In [ ]:
con.execute(r"""
CREATE OR REPLACE TABLE s3_name_token_index AS
SELECT DISTINCT
    entity_id,
    token
FROM s3_normalized,
LATERAL unnest(
    string_split(normalized_name, ' ')
) AS t(token)
WHERE
    token != ''
    AND token IN (
        SELECT token
        FROM source3_token_frequency
        WHERE record_count <= 10000
    )
""")

print("S3 name-token index created.")

S3 name-token index created.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s2_name_token_index_d AS
SELECT DISTINCT
    entity_id,
    token
FROM s2_name_token_index
""")

print("Deduplicated S2 name-token index created.")

Deduplicated S2 name-token index created.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s1_name_token_index_d AS
SELECT DISTINCT
    entity_id,
    token
FROM s1_name_token_index
""")

print("Deduplicated S1 name-token index created.")

Deduplicated S1 name-token index created.


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW s1_name_tokens_ranked_s2 AS
SELECT
    s1.entity_id,
    s1.token,
    tf.record_count AS target_frequency,

    ROW_NUMBER() OVER (
        PARTITION BY s1.entity_id
        ORDER BY tf.record_count ASC, s1.token
    ) AS token_rank

FROM s1_name_token_index_d s1

JOIN source2_token_frequency tf
    ON s1.token = tf.token

WHERE tf.record_count <= 10000
""")

print("S1 name tokens ranked by S2 rarity.")

S1 name tokens ranked by S2 rarity.


In [ ]:
con.execute("""
SELECT *
FROM s1_name_tokens_ranked_s2
ORDER BY entity_id, token_rank
LIMIT 30
""").df()

,entity_id,token,target_frequency,token_rank
0,S1-100000221,redwood,1783,1
1,S1-100001535,bangalore,8878,1
2,S1-100001665,future,4565,1
3,S1-100002411,indo,3187,1
4,S1-100002771,rameshwar,68,1
5,S1-100002771,projects,9611,2
6,S1-100003009,oyw,9,1
7,S1-100003009,velocity,130,2
8,S1-100003333,stag,1058,1
9,S1-100003736,thane,2606,1


In [ ]:
top1_name_s2 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs
FROM (
    SELECT
        entity_id,
        token
    FROM s1_name_tokens_ranked_s2
    WHERE token_rank = 1
) s1_tokens

JOIN s2_name_token_index_d s2_tokens
    ON s1_tokens.token = s2_tokens.token

JOIN s1_normalized s1
    ON s1.entity_id = s1_tokens.entity_id

JOIN s2_normalized s2
    ON s2.entity_id = s2_tokens.entity_id

WHERE s1.country = s2.country
""").fetchone()[0]

print("Top-1 name-token S1-S2 candidate pairs:", top1_name_s2)
print(
    "Average candidates per S1:",
    top1_name_s2 / 2206821
)

Top-1 name-token S1-S2 candidate pairs: 2591568755
Average candidates per S1: 1174.3447950694688


In [ ]:
top1_name_recall_s2 = con.execute("""
WITH true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S2-%'
),

top1_tokens AS (
    SELECT
        entity_id,
        token
    FROM s1_name_tokens_ranked_s2
    WHERE token_rank = 1
)

SELECT
    COUNT(*) AS total_matches,

    SUM(
        CASE
            WHEN EXISTS (
                SELECT 1
                FROM s2_name_token_index_d s2t
                WHERE s2t.entity_id = gt.matched_entity_id
                  AND s2t.token = top1.token
            )
            AND s1.country = s2.country
            THEN 1
            ELSE 0
        END
    ) AS captured

FROM true_pairs gt

JOIN top1_tokens top1
    ON gt.source1_entity_id = top1.entity_id

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_normalized s2
    ON gt.matched_entity_id = s2.entity_id
""").df()

top1_name_recall_s2

,total_matches,captured
0,3259297,2502259.0


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s2_name_token_country_frequency AS
SELECT
    s2.country,
    s2t.token,
    COUNT(*) AS record_count
FROM s2_normalized s2
JOIN s2_name_token_index_d s2t
    ON s2.entity_id = s2t.entity_id
GROUP BY
    s2.country,
    s2t.token
""")

print("S2 country-token frequencies created.")

S2 country-token frequencies created.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s3_name_token_country_frequency AS
SELECT
    s3.country,
    s3t.token,
    COUNT(*) AS record_count
FROM s3_normalized s3
JOIN s3_name_token_index s3t
    ON s3.entity_id = s3t.entity_id
GROUP BY
    s3.country,
    s3t.token
""")

print("S3 country-token frequencies created.")

S3 country-token frequencies created.


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW s1_name_tokens_ranked_s2_country AS
SELECT
    s1.entity_id,
    s1.country,
    s1t.token,
    tf.record_count AS target_country_frequency,

    ROW_NUMBER() OVER (
        PARTITION BY s1.entity_id
        ORDER BY tf.record_count ASC, s1t.token
    ) AS token_rank

FROM s1_normalized s1

JOIN s1_name_token_index_d s1t
    ON s1.entity_id = s1t.entity_id

JOIN s2_name_token_country_frequency tf
    ON s1.country = tf.country
   AND s1t.token = tf.token

WHERE tf.record_count <= 10000
""")

print("Country-aware S1 name ranking created.")

Country-aware S1 name ranking created.


In [ ]:
con.execute("""
SELECT *
FROM s1_name_tokens_ranked_s2_country
ORDER BY entity_id, token_rank
LIMIT 30
""").df()

,entity_id,country,token,target_country_frequency,token_rank
0,S1-100000221,US,redwood,1741,1
1,S1-100001535,India,bangalore,8795,1
2,S1-100001665,India,future,3680,1
3,S1-100002411,India,indo,3167,1
4,S1-100002771,India,rameshwar,67,1
5,S1-100002771,India,projects,9535,2
6,S1-100003009,India,oyw,9,1
7,S1-100003009,India,velocity,129,2
8,S1-100003333,US,stag,1023,1
9,S1-100003736,India,thane,2581,1


In [ ]:
top1_country_candidate_volume_s2 = con.execute("""
SELECT
    SUM(tf.record_count) AS candidate_pairs
FROM s1_name_tokens_ranked_s2_country r

JOIN s2_name_token_country_frequency tf
    ON r.country = tf.country
   AND r.token = tf.token

WHERE r.token_rank = 1
""").fetchone()[0]

print(
    "Country-aware Top-1 S1-S2 candidate pairs:",
    int(top1_country_candidate_volume_s2)
)

print(
    "Average candidates per S1:",
    int(top1_country_candidate_volume_s2) / 2206821
)

Country-aware Top-1 S1-S2 candidate pairs: 2552494797
Average candidates per S1: 1156.6388016971018


In [ ]:
coverage_s2 = con.execute("""
SELECT
    COUNT(DISTINCT entity_id) AS entities_with_eligible_token
FROM s1_name_tokens_ranked_s2_country
WHERE token_rank = 1
""").fetchone()[0]

print("S1 entities with eligible top-1 token:", coverage_s2)
print("Total S1 entities:", 2206821)
print(
    "Coverage:",
    f"{coverage_s2 / 2206821:.2%}"
)

S1 entities with eligible top-1 token: 1945395
Total S1 entities: 2206821
Coverage: 88.15%


In [ ]:
frequency_caps = [100, 250, 500, 1000, 2000, 5000, 10000]

top1_name_volume_results = []

for cap in frequency_caps:

    result = con.execute(f"""
    WITH ranked AS (
        SELECT
            s1.entity_id,
            s1.country,
            s1t.token,
            tf.record_count,

            ROW_NUMBER() OVER (
                PARTITION BY s1.entity_id
                ORDER BY tf.record_count ASC, s1t.token
            ) AS token_rank

        FROM s1_normalized s1

        JOIN s1_name_token_index_d s1t
            ON s1.entity_id = s1t.entity_id

        JOIN s2_name_token_country_frequency tf
            ON s1.country = tf.country
           AND s1t.token = tf.token

        WHERE tf.record_count <= {cap}
    )

    SELECT
        COUNT(*) AS s1_with_token,
        COALESCE(SUM(record_count), 0) AS candidate_pairs
    FROM ranked
    WHERE token_rank = 1
    """).fetchone()

    s1_with_token = result[0]
    candidate_pairs = result[1]

    top1_name_volume_results.append({
        "frequency_cap": cap,
        "s1_with_token": s1_with_token,
        "coverage": s1_with_token / 2206821,
        "candidate_pairs": candidate_pairs,
        "avg_candidates_per_s1": candidate_pairs / 2206821
    })

pd.DataFrame(top1_name_volume_results)

,frequency_cap,s1_with_token,coverage,candidate_pairs,avg_candidates_per_s1
0,100,772707,0.350145,31578289,14.309402
1,250,988899,0.448110,64812143,29.369008
2,500,1088846,0.493400,100164498,45.388592
3,1000,1348361,0.610997,306232957,138.766559
4,2000,1431186,0.648528,425269304,192.706751
5,5000,1839447,0.833528,1839335129,833.477264
6,10000,1945395,0.881537,2552494797,1156.638802


In [ ]:
exact_address_s2 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs
FROM s1_normalized s1
JOIN s2_normalized s2
    ON s1.country = s2.country
   AND s1.normalized_address = s2.normalized_address
WHERE
    s1.normalized_address != ''
    AND s2.normalized_address != ''
""").fetchone()[0]

print("S1-S2 exact-address candidate pairs:", exact_address_s2)
print(
    "Average candidates per S1:",
    exact_address_s2 / 2206821
)

S1-S2 exact-address candidate pairs: 527396
Average candidates per S1: 0.23898449398478627


In [ ]:
exact_address_s3 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs
FROM s1_normalized s1
JOIN s3_normalized s3
    ON s1.country = s3.country
   AND s1.normalized_address = s3.normalized_address
WHERE
    s1.normalized_address != ''
    AND s3.normalized_address != ''
""").fetchone()[0]

print("S1-S3 exact-address candidate pairs:", exact_address_s3)
print(
    "Average candidates per S1:",
    exact_address_s3 / 2206821
)

S1-S3 exact-address candidate pairs: 205504
Average candidates per S1: 0.09312218797990413


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s1_address_token_index_d AS
SELECT DISTINCT
    entity_id,
    token
FROM s1_address_token_index
WHERE token != ''
""")

print("S1 deduplicated address-token index created.")

S1 deduplicated address-token index created.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s2_address_token_index_d AS
SELECT DISTINCT
    entity_id,
    token
FROM s2_address_token_index
WHERE token != ''
""")

print("S2 deduplicated address-token index created.")

S2 deduplicated address-token index created.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s3_address_token_index AS
SELECT DISTINCT
    entity_id,
    token
FROM s3_normalized,
LATERAL unnest(
    string_split(normalized_address, ' ')
) AS t(token)
WHERE
    token != ''
    AND token IN (
        SELECT token
        FROM source3_address_token_frequency
        WHERE record_count <= 10000
    )
""")

print("S3 address-token index created.")

S3 address-token index created.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s1_address_token_index_d AS
SELECT DISTINCT
    entity_id,
    token
FROM s1_address_token_index
WHERE token != ''
""")

con.execute("""
CREATE OR REPLACE TABLE s2_address_token_index_d AS
SELECT DISTINCT
    entity_id,
    token
FROM s2_address_token_index
WHERE token != ''
""")

print("Address indexes ready.")

Address indexes ready.


In [ ]:
print(
    "S1 address index:",
    con.execute(
        "SELECT COUNT(*) FROM s1_address_token_index_d"
    ).fetchone()[0]
)

print(
    "S2 address index:",
    con.execute(
        "SELECT COUNT(*) FROM s2_address_token_index_d"
    ).fetchone()[0]
)

print(
    "S3 address index:",
    con.execute(
        "SELECT COUNT(*) FROM s3_address_token_index"
    ).fetchone()[0]
)

S1 address index: 18021687
S2 address index: 16457287
S3 address index: 16406037


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s2_address_token_country_frequency AS
SELECT
    s2.country,
    s2t.token,
    COUNT(*) AS record_count
FROM s2_normalized s2
JOIN s2_address_token_index_d s2t
    ON s2.entity_id = s2t.entity_id
GROUP BY
    s2.country,
    s2t.token
""")

print("S2 country-address frequencies created.")

S2 country-address frequencies created.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s3_address_token_country_frequency AS
SELECT
    s3.country,
    s3t.token,
    COUNT(*) AS record_count
FROM s3_normalized s3
JOIN s3_address_token_index s3t
    ON s3.entity_id = s3t.entity_id
GROUP BY
    s3.country,
    s3t.token
""")

print("S3 country-address frequencies created.")

S3 country-address frequencies created.


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW s1_address_tokens_ranked_s2 AS
SELECT
    s1.entity_id,
    s1.country,
    s1t.token,
    tf.record_count AS target_country_frequency,

    ROW_NUMBER() OVER (
        PARTITION BY s1.entity_id
        ORDER BY tf.record_count ASC, s1t.token
    ) AS token_rank

FROM s1_normalized s1

JOIN s1_address_token_index_d s1t
    ON s1.entity_id = s1t.entity_id

JOIN s2_address_token_country_frequency tf
    ON s1.country = tf.country
   AND s1t.token = tf.token

WHERE tf.record_count <= 10000
""")

print("S1 address tokens ranked for S2.")

S1 address tokens ranked for S2.


In [ ]:
con.execute("""
SELECT *
FROM s1_address_tokens_ranked_s2
ORDER BY entity_id, token_rank
LIMIT 30
""").df()

,entity_id,country,token,target_country_frequency,token_rank
0,S1-100000221,US,526,1006,1
1,S1-100000221,US,cary,3258,2
2,S1-100000221,US,walnut,6636,3
3,S1-100001512,India,135,2893,1
4,S1-100001512,India,durga,3000,2
5,S1-100001512,India,raj,6213,3
6,S1-100001535,India,songbirds,29,1
7,S1-100001535,India,skyi,30,2
8,S1-100001535,India,bhugaon,117,3
9,S1-100001535,India,lawns,209,4


In [ ]:
top1_address_volume_s2 = con.execute("""
SELECT
    COALESCE(SUM(r.target_country_frequency), 0) AS candidate_pairs
FROM s1_address_tokens_ranked_s2 r
WHERE r.token_rank = 1
""").fetchone()[0]

print(
    "Top-1 address S1-S2 candidate pairs:",
    int(top1_address_volume_s2)
)

print(
    "Average candidates per S1:",
    int(top1_address_volume_s2) / 2206821
)

Top-1 address S1-S2 candidate pairs: 747920259
Average candidates per S1: 338.9129698330766


In [ ]:
top1_address_coverage_s2 = con.execute("""
SELECT
    COUNT(DISTINCT entity_id)
FROM s1_address_tokens_ranked_s2
WHERE token_rank = 1
""").fetchone()[0]

print(
    "S1 entities with Top-1 address token:",
    top1_address_coverage_s2
)

print(
    "S1 coverage:",
    f"{top1_address_coverage_s2 / 2206821:.2%}"
)

S1 entities with Top-1 address token: 2201424
S1 coverage: 99.76%


In [ ]:
top1_address_recall_s2 = con.execute("""
WITH true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S2-%'
),

top1 AS (
    SELECT
        entity_id,
        country,
        token
    FROM s1_address_tokens_ranked_s2
    WHERE token_rank = 1
)

SELECT
    COUNT(*) AS total_true_matches,

    SUM(
        CASE
            WHEN top1.token IS NOT NULL
             AND s1.country = s2.country
             AND s2idx.entity_id IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS captured_matches

FROM true_pairs gt

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_normalized s2
    ON gt.matched_entity_id = s2.entity_id

LEFT JOIN top1
    ON gt.source1_entity_id = top1.entity_id

LEFT JOIN s2_address_token_index_d s2idx
    ON s2idx.entity_id = gt.matched_entity_id
   AND s2idx.token = top1.token
""").df()

top1_address_recall_s2

,total_true_matches,captured_matches
0,3693619,3008482.0


In [ ]:
top1_total = int(
    top1_address_recall_s2.loc[0, "total_true_matches"]
)

top1_captured = int(
    top1_address_recall_s2.loc[0, "captured_matches"]
)

print("Total S2 true matches:", top1_total)
print("Top-1 address matches captured:", top1_captured)

print(
    "Top-1 address overall recall:",
    f"{top1_captured / top1_total:.4%}"
)

Total S2 true matches: 3693619
Top-1 address matches captured: 3008482
Top-1 address overall recall: 81.4508%


In [ ]:
top2_address_volume_s2 = con.execute("""
WITH top2 AS (
    SELECT
        entity_id,
        country,

        MAX(
            CASE
                WHEN token_rank = 1 THEN token
            END
        ) AS token1,

        MAX(
            CASE
                WHEN token_rank = 2 THEN token
            END
        ) AS token2

    FROM s1_address_tokens_ranked_s2

    WHERE token_rank <= 2

    GROUP BY
        entity_id,
        country

    HAVING COUNT(*) = 2
)

SELECT
    COUNT(DISTINCT top2.entity_id) AS s1_with_two_tokens,

    COUNT(*) AS candidate_pairs

FROM top2

JOIN s2_address_token_index_d a
    ON a.token = top2.token1

JOIN s2_address_token_index_d b
    ON b.entity_id = a.entity_id
   AND b.token = top2.token2

JOIN s2_normalized s2
    ON s2.entity_id = a.entity_id
   AND s2.country = top2.country
""").df()

top2_address_volume_s2

,s1_with_two_tokens,candidate_pairs
0,1683723,18922448


In [ ]:
s1_two_token = int(
    top2_address_volume_s2.loc[0, "s1_with_two_tokens"]
)

top2_candidates = int(
    top2_address_volume_s2.loc[0, "candidate_pairs"]
)

print("S1 entities with 2 eligible address tokens:", s1_two_token)

print(
    "S1 coverage:",
    f"{s1_two_token / 2206821:.2%}"
)

print(
    "Top-2 address candidate pairs:",
    top2_candidates
)

print(
    "Average candidates per S1:",
    top2_candidates / 2206821
)

print(
    "Average candidates per covered S1:",
    top2_candidates / s1_two_token
)

S1 entities with 2 eligible address tokens: 1683723
S1 coverage: 76.30%
Top-2 address candidate pairs: 18922448
Average candidates per S1: 8.574527793599934
Average candidates per covered S1: 11.23845668200767


In [ ]:
top2_address_recall_s2 = con.execute("""
WITH true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S2-%'
),

top2 AS (
    SELECT
        entity_id,
        country,

        MAX(
            CASE
                WHEN token_rank = 1 THEN token
            END
        ) AS token1,

        MAX(
            CASE
                WHEN token_rank = 2 THEN token
            END
        ) AS token2

    FROM s1_address_tokens_ranked_s2

    WHERE token_rank <= 2

    GROUP BY
        entity_id,
        country

    HAVING COUNT(*) = 2
)

SELECT
    COUNT(*) AS total_true_matches,

    SUM(
        CASE
            WHEN top2.token1 IS NOT NULL
             AND top2.token2 IS NOT NULL
             AND s1.country = s2.country
             AND a.entity_id IS NOT NULL
             AND b.entity_id IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS captured_matches

FROM true_pairs gt

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_normalized s2
    ON gt.matched_entity_id = s2.entity_id

LEFT JOIN top2
    ON gt.source1_entity_id = top2.entity_id

LEFT JOIN s2_address_token_index_d a
    ON a.entity_id = gt.matched_entity_id
   AND a.token = top2.token1

LEFT JOIN s2_address_token_index_d b
    ON b.entity_id = gt.matched_entity_id
   AND b.token = top2.token2
""").df()

top2_address_recall_s2

,total_true_matches,captured_matches
0,3693619,2529798.0


In [ ]:
top2_total = int(
    top2_address_recall_s2.loc[0, "total_true_matches"]
)

top2_captured = int(
    top2_address_recall_s2.loc[0, "captured_matches"]
)

print("Total S2 true matches:", top2_total)
print("Top-2 address matches captured:", top2_captured)

print(
    "Top-2 address overall recall:",
    f"{top2_captured / top2_total:.4%}"
)

Total S2 true matches: 3693619
Top-2 address matches captured: 2529798
Top-2 address overall recall: 68.4910%


In [ ]:
combined_name_top2addr_s2 = con.execute("""
WITH
true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S2-%'
),

name_token_candidates AS (
    SELECT DISTINCT
        s1.entity_id AS source1_entity_id,
        s2t.entity_id AS candidate_entity_id
    FROM s1_normalized s1

    JOIN s1_name_token_index_d s1t
        ON s1.entity_id = s1t.entity_id

    JOIN s2_name_token_index_d s2t
        ON s1t.token = s2t.token

    JOIN s2_normalized s2
        ON s2.entity_id = s2t.entity_id
       AND s1.country = s2.country

    WHERE s1t.token IN (
        SELECT token
        FROM source2_token_frequency
        WHERE record_count <= 10000
    )
),

exact_name_candidates AS (
    SELECT
        s1.entity_id AS source1_entity_id,
        s2.entity_id AS candidate_entity_id
    FROM s1_normalized s1
    JOIN s2_normalized s2
        ON s1.country = s2.country
       AND s1.normalized_name = s2.normalized_name
    WHERE
        s1.normalized_name != ''
        AND s2.normalized_name != ''
),

top2_address AS (
    SELECT
        entity_id,
        country,

        MAX(
            CASE WHEN token_rank = 1 THEN token END
        ) AS token1,

        MAX(
            CASE WHEN token_rank = 2 THEN token END
        ) AS token2

    FROM s1_address_tokens_ranked_s2

    WHERE token_rank <= 2

    GROUP BY
        entity_id,
        country

    HAVING COUNT(*) = 2
),

top2_address_candidates AS (
    SELECT DISTINCT
        a.entity_id AS source1_entity_id,
        b.entity_id AS candidate_entity_id

    FROM top2_address

    JOIN s2_address_token_index_d a
        ON a.token = top2_address.token1

    JOIN s2_address_token_index_d b
        ON b.entity_id = a.entity_id
       AND b.token = top2_address.token2

    JOIN s2_normalized s2
        ON s2.entity_id = a.entity_id
       AND s2.country = top2_address.country
),

all_candidates AS (
    SELECT * FROM name_token_candidates

    UNION

    SELECT * FROM exact_name_candidates

    UNION

    SELECT * FROM top2_address_candidates
)

SELECT
    COUNT(*) AS total_true_matches,

    SUM(
        CASE
            WHEN ac.candidate_entity_id IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS captured_matches

FROM true_pairs gt

LEFT JOIN all_candidates ac
    ON gt.source1_entity_id = ac.source1_entity_id
   AND gt.matched_entity_id = ac.candidate_entity_id
""").df()

combined_name_top2addr_s2

OutOfMemoryException: Out of Memory Error: failed to offload data block of size 256.0 KiB (128.4 GiB/128.4 GiB used).
This limit was set by the 'max_temp_directory_size' setting.
By default, this setting utilizes the available disk space on the drive where the 'temp_directory' is located.
You can adjust this setting, by using (for example) PRAGMA max_temp_directory_size='10GiB'

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [ ]:
exact_address_recall_s2 = con.execute("""
SELECT
    COUNT(*) AS total_true_matches,

    SUM(
        CASE
            WHEN s1.normalized_address != ''
             AND s2.normalized_address != ''
             AND s1.country = s2.country
             AND s1.normalized_address = s2.normalized_address
            THEN 1
            ELSE 0
        END
    ) AS captured_matches

FROM ground_truth_matches gt

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_normalized s2
    ON gt.matched_entity_id = s2.entity_id

WHERE gt.matched_entity_id LIKE 'S2-%'
""").df()

exact_address_recall_s2

,total_true_matches,captured_matches
0,3693619,430971.0


In [ ]:
total = int(
    exact_address_recall_s2.loc[0, "total_true_matches"]
)

captured = int(
    exact_address_recall_s2.loc[0, "captured_matches"]
)

print("S2 true matches:", total)
print("S2 exact-address matches captured:", captured)

print(
    "S2 exact-address recall:",
    f"{captured / total:.4%}"
)

S2 true matches: 3693619
S2 exact-address matches captured: 430971
S2 exact-address recall: 11.6680%


In [ ]:
exact_address_volume_s2 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs
FROM s1_normalized s1

JOIN s2_normalized s2
    ON s1.country = s2.country
   AND s1.normalized_address = s2.normalized_address

WHERE
    s1.normalized_address != ''
    AND s2.normalized_address != ''
""").fetchone()[0]

print(
    "S2 exact-address candidate pairs:",
    exact_address_volume_s2
)

print(
    "Average candidates per S1:",
    exact_address_volume_s2 / 2206821
)

S2 exact-address candidate pairs: 527396
Average candidates per S1: 0.23898449398478627


In [ ]:
exact_address_recall_s3 = con.execute("""
SELECT
    COUNT(*) AS total_true_matches,

    SUM(
        CASE
            WHEN s1.normalized_address != ''
             AND s3.normalized_address != ''
             AND s1.country = s3.country
             AND s1.normalized_address = s3.normalized_address
            THEN 1
            ELSE 0
        END
    ) AS captured_matches

FROM ground_truth_matches gt

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s3_normalized s3
    ON gt.matched_entity_id = s3.entity_id

WHERE gt.matched_entity_id LIKE 'S3-%'
""").df()

exact_address_recall_s3

,total_true_matches,captured_matches
0,3944746,170437.0


In [ ]:
total_s3 = int(
    exact_address_recall_s3.loc[0, "total_true_matches"]
)

captured_s3 = int(
    exact_address_recall_s3.loc[0, "captured_matches"]
)

print("S3 true matches:", total_s3)
print("S3 exact-address matches captured:", captured_s3)

print(
    "S3 exact-address recall:",
    f"{captured_s3 / total_s3:.4%}"
)

S3 true matches: 3944746
S3 exact-address matches captured: 170437
S3 exact-address recall: 4.3206%


In [ ]:
exact_address_volume_s3 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs
FROM s1_normalized s1

JOIN s3_normalized s3
    ON s1.country = s3.country
   AND s1.normalized_address = s3.normalized_address

WHERE
    s1.normalized_address != ''
    AND s3.normalized_address != ''
""").fetchone()[0]

print(
    "S1-S3 exact-address candidate pairs:",
    exact_address_volume_s3
)

print(
    "Average candidates per S1:",
    exact_address_volume_s3 / 2206821
)

S1-S3 exact-address candidate pairs: 205504
Average candidates per S1: 0.09312218797990413


In [ ]:
top2_name_volume_s2 = con.execute("""
WITH top2 AS (
    SELECT
        entity_id,
        country,

        MAX(
            CASE
                WHEN token_rank = 1 THEN token
            END
        ) AS token1,

        MAX(
            CASE
                WHEN token_rank = 2 THEN token
            END
        ) AS token2

    FROM s1_name_tokens_ranked_s2_country

    WHERE token_rank <= 2

    GROUP BY
        entity_id,
        country

    HAVING COUNT(*) = 2
)

SELECT
    COUNT(DISTINCT top2.entity_id) AS s1_with_two_tokens,
    COUNT(*) AS candidate_pairs

FROM top2

JOIN s2_name_token_index_d a
    ON a.token = top2.token1

JOIN s2_name_token_index_d b
    ON b.entity_id = a.entity_id
   AND b.token = top2.token2

JOIN s2_normalized s2
    ON s2.entity_id = a.entity_id
   AND s2.country = top2.country
""").df()

top2_name_volume_s2

OutOfMemoryException: Out of Memory Error: failed to pin block of size 256.0 KiB (6.2 GiB/6.2 GiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

In [ ]:
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

print("DuckDB configured for lower-memory joins.")

DuckDB configured for lower-memory joins.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s2_name_token_pair_index AS

SELECT
    s2.country,
    a.entity_id,
    a.token AS token_a,
    b.token AS token_b

FROM s2_name_token_index_d a

JOIN s2_name_token_index_d b
    ON a.entity_id = b.entity_id
   AND a.token < b.token

JOIN s2_normalized s2
    ON s2.entity_id = a.entity_id
""")

print("S2 two-token index created.")

S2 two-token index created.


In [ ]:
s2_pair_count = con.execute("""
SELECT COUNT(*)
FROM s2_name_token_pair_index
""").fetchone()[0]

print("S2 name-token pair rows:", s2_pair_count)

S2 name-token pair rows: 4925030


In [ ]:
con.execute("""
SELECT *
FROM s2_name_token_pair_index
LIMIT 20
""").df()

,country,entity_id,token_a,token_b
0,US,S2-285761393,advisory,meex
1,US,S2-138807660,riverside,southampton
2,India,S2-678172790,fóundation,rameshwaram
3,US,S2-416950492,fortress,mabelle
4,US,S2-425764694,jones,layla
5,India,S2-674387405,samiti,sbgme
6,US,S2-544796034,cho,galaxyedge
7,US,S2-986972141,book,store
8,India,S2-294818521,मैनेजमेंट,सदर्न
9,India,S2-375556685,maharashtra,rehabilitation


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW s1_name_top2_s2 AS

SELECT
    entity_id,
    country,

    MAX(
        CASE
            WHEN token_rank = 1 THEN token
        END
    ) AS token1,

    MAX(
        CASE
            WHEN token_rank = 2 THEN token
        END
    ) AS token2

FROM s1_name_tokens_ranked_s2_country

WHERE token_rank <= 2

GROUP BY
    entity_id,
    country

HAVING COUNT(*) = 2
""")

print("S1 Top-2 name keys created.")

S1 Top-2 name keys created.


In [ ]:
con.execute("""
SELECT *
FROM s1_name_top2_s2
LIMIT 20
""").df()

,entity_id,country,token1,token2
0,S1-100476606,US,ioq,mustang
1,S1-101127593,India,folk,suryavi
2,S1-102134868,India,surya,impex
3,S1-102256434,US,wiley,person
4,S1-10249653,US,plants,chin
5,S1-102598186,India,bhutoria,mutual
6,S1-103303734,US,bogucki,vanegas
7,S1-103452251,US,winvest,king
8,S1-103514223,India,charming,doorstep
9,S1-103557640,India,galaxy,lakshmi


In [ ]:
s1_name_top2_count = con.execute("""
SELECT COUNT(*)
FROM s1_name_top2_s2
""").fetchone()[0]

print("S1 entities with Top-2 eligible name tokens:", s1_name_top2_count)

print(
    "Coverage:",
    f"{s1_name_top2_count / 2206821:.2%}"
)

S1 entities with Top-2 eligible name tokens: 1088183
Coverage: 49.31%


In [ ]:
s1_name_top2_count = con.execute("""
SELECT COUNT(*)
FROM s1_name_top2_s2
""").fetchone()[0]

print("S1 entities with Top-2 eligible name tokens:", s1_name_top2_count)

print(
    "Coverage:",
    f"{s1_name_top2_count / 2206821:.2%}"
)

S1 entities with Top-2 eligible name tokens: 1088183
Coverage: 49.31%


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW s1_name_top2_s2_key AS

SELECT
    entity_id,
    country,

    LEAST(token1, token2) AS token_a,
    GREATEST(token1, token2) AS token_b

FROM s1_name_top2_s2
""")

print("Composite S1 name keys created.")

Composite S1 name keys created.


In [ ]:
top2_name_volume_s2 = con.execute("""
SELECT
    COUNT(*) AS candidate_pairs

FROM s1_name_top2_s2_key s1

JOIN s2_name_token_pair_index s2
    ON s1.country = s2.country
   AND s1.token_a = s2.token_a
   AND s1.token_b = s2.token_b
""").fetchone()[0]

print(
    "Top-2 name S1-S2 candidate pairs:",
    top2_name_volume_s2
)

print(
    "Average candidates per S1:",
    top2_name_volume_s2 / 2206821
)

print(
    "Average candidates per covered S1:",
    top2_name_volume_s2 / s1_name_top2_count
)

Top-2 name S1-S2 candidate pairs: 61548114
Average candidates per S1: 27.88994395104995
Average candidates per covered S1: 56.560444337027874


In [ ]:
top2_name_recall_s2 = con.execute("""
WITH true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S2-%'
),

top2 AS (
    SELECT
        entity_id,
        country,
        token_a,
        token_b
    FROM s1_name_top2_s2_key
)

SELECT
    COUNT(*) AS total_true_matches,

    SUM(
        CASE
            WHEN s2p.entity_id IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS captured_matches

FROM true_pairs gt

LEFT JOIN top2
    ON gt.source1_entity_id = top2.entity_id

LEFT JOIN s2_name_token_pair_index s2p
    ON s2p.entity_id = gt.matched_entity_id
   AND s2p.country = top2.country
   AND s2p.token_a = top2.token_a
   AND s2p.token_b = top2.token_b
""").df()

top2_name_recall_s2

,total_true_matches,captured_matches
0,3693619,1227171.0


In [ ]:
total = int(
    top2_name_recall_s2.loc[0, "total_true_matches"]
)

captured = int(
    top2_name_recall_s2.loc[0, "captured_matches"]
)

print("S2 true matches:", total)
print("S2 Top-2 name matches captured:", captured)

print(
    "S2 Top-2 name recall:",
    f"{captured / total:.4%}"
)

S2 true matches: 3693619
S2 Top-2 name matches captured: 1227171
S2 Top-2 name recall: 33.2241%


In [ ]:
combined_small_blocks_s2 = con.execute("""
WITH
true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S2-%'
),

top2_name AS (
    SELECT
        entity_id,
        country,
        token_a,
        token_b
    FROM s1_name_top2_s2_key
),

top2_address AS (
    SELECT
        entity_id,
        country,

        MAX(
            CASE WHEN token_rank = 1 THEN token END
        ) AS token1,

        MAX(
            CASE WHEN token_rank = 2 THEN token END
        ) AS token2

    FROM s1_address_tokens_ranked_s2

    WHERE token_rank <= 2

    GROUP BY
        entity_id,
        country

    HAVING COUNT(*) = 2
)

SELECT
    COUNT(*) AS total_true_matches,

    SUM(
        CASE
            WHEN
                /* Exact normalized name */
                (
                    s1.normalized_name != ''
                    AND
                    s1.normalized_name = s2.normalized_name
                )

                OR

                /* Exact normalized address */
                (
                    s1.normalized_address != ''
                    AND
                    s1.normalized_address = s2.normalized_address
                )

                OR

                /* Top-2 name */
                EXISTS (
                    SELECT 1
                    FROM top2_name tn
                    JOIN s2_name_token_pair_index np
                        ON np.entity_id = gt.matched_entity_id
                       AND np.country = tn.country
                       AND np.token_a = tn.token_a
                       AND np.token_b = tn.token_b
                    WHERE tn.entity_id = gt.source1_entity_id
                )

                OR

                /* Top-2 address */
                EXISTS (
                    SELECT 1
                    FROM top2_address ta

                    WHERE
                        ta.entity_id = gt.source1_entity_id
                        AND ta.country = s2.country

                        AND EXISTS (
                            SELECT 1
                            FROM s2_address_token_index_d a
                            WHERE
                                a.entity_id = gt.matched_entity_id
                                AND a.token = ta.token1
                        )

                        AND EXISTS (
                            SELECT 1
                            FROM s2_address_token_index_d b
                            WHERE
                                b.entity_id = gt.matched_entity_id
                                AND b.token = ta.token2
                        )
                )

            THEN 1
            ELSE 0
        END
    ) AS captured_matches

FROM true_pairs gt

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_normalized s2
    ON gt.matched_entity_id = s2.entity_id
""").df()

combined_small_blocks_s2

,total_true_matches,captured_matches
0,3693619,3029621.0


In [ ]:
total = int(
    combined_small_blocks_s2.loc[0, "total_true_matches"]
)

captured = int(
    combined_small_blocks_s2.loc[0, "captured_matches"]
)

print("S2 total true matches:", total)
print("S2 captured by combined blocks:", captured)

print(
    "S2 combined recall:",
    f"{captured / total:.4%}"
)

S2 total true matches: 3693619
S2 captured by combined blocks: 3029621
S2 combined recall: 82.0231%


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW s1_name_tokens_ranked_s3_country AS
SELECT
    s1.entity_id,
    s1.country,
    s1t.token,
    tf.record_count AS target_country_frequency,

    ROW_NUMBER() OVER (
        PARTITION BY s1.entity_id
        ORDER BY tf.record_count ASC, s1t.token
    ) AS token_rank

FROM s1_normalized s1

JOIN s1_name_token_index_d s1t
    ON s1.entity_id = s1t.entity_id

JOIN s3_name_token_country_frequency tf
    ON s1.country = tf.country
   AND s1t.token = tf.token

WHERE tf.record_count <= 10000
""")

print("S1 name tokens ranked for S3.")

S1 name tokens ranked for S3.


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW s1_name_top2_s3 AS

SELECT
    entity_id,
    country,

    LEAST(
        MAX(CASE WHEN token_rank = 1 THEN token END),
        MAX(CASE WHEN token_rank = 2 THEN token END)
    ) AS token_a,

    GREATEST(
        MAX(CASE WHEN token_rank = 1 THEN token END),
        MAX(CASE WHEN token_rank = 2 THEN token END)
    ) AS token_b

FROM s1_name_tokens_ranked_s3_country

WHERE token_rank <= 2

GROUP BY
    entity_id,
    country

HAVING COUNT(*) = 2
""")

print("S1 Top-2 name keys for S3 created.")

S1 Top-2 name keys for S3 created.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE s3_name_token_pair_index AS

SELECT
    s3.country,
    a.entity_id,
    a.token AS token_a,
    b.token AS token_b

FROM s3_name_token_index a

JOIN s3_name_token_index b
    ON a.entity_id = b.entity_id
   AND a.token < b.token

JOIN s3_normalized s3
    ON s3.entity_id = a.entity_id
""")

print("S3 two-token name index created.")

S3 two-token name index created.


In [ ]:
con.execute("""
CREATE OR REPLACE VIEW s1_address_tokens_ranked_s3 AS
SELECT
    s1.entity_id,
    s1.country,
    s1t.token,
    tf.record_count AS target_country_frequency,

    ROW_NUMBER() OVER (
        PARTITION BY s1.entity_id
        ORDER BY tf.record_count ASC, s1t.token
    ) AS token_rank

FROM s1_normalized s1

JOIN s1_address_token_index_d s1t
    ON s1.entity_id = s1t.entity_id

JOIN s3_address_token_country_frequency tf
    ON s1.country = tf.country
   AND s1t.token = tf.token

WHERE tf.record_count <= 10000
""")

print("S1 address tokens ranked for S3.")

S1 address tokens ranked for S3.


In [ ]:
combined_small_blocks_s3 = con.execute("""
WITH
true_pairs AS (
    SELECT
        source1_entity_id,
        matched_entity_id
    FROM ground_truth_matches
    WHERE matched_entity_id LIKE 'S3-%'
),

top2_name AS (
    SELECT
        entity_id,
        country,
        token_a,
        token_b
    FROM s1_name_top2_s3
),

top2_address AS (
    SELECT
        entity_id,
        country,

        MAX(
            CASE WHEN token_rank = 1 THEN token END
        ) AS token1,

        MAX(
            CASE WHEN token_rank = 2 THEN token END
        ) AS token2

    FROM s1_address_tokens_ranked_s3

    WHERE token_rank <= 2

    GROUP BY
        entity_id,
        country

    HAVING COUNT(*) = 2
)

SELECT
    COUNT(*) AS total_true_matches,

    SUM(
        CASE
            WHEN
                (
                    s1.normalized_name != ''
                    AND
                    s1.normalized_name = s3.normalized_name
                )

                OR

                (
                    s1.normalized_address != ''
                    AND
                    s1.normalized_address = s3.normalized_address
                )

                OR

                EXISTS (
                    SELECT 1
                    FROM top2_name tn

                    JOIN s3_name_token_pair_index np
                        ON np.entity_id = gt.matched_entity_id
                       AND np.country = tn.country
                       AND np.token_a = tn.token_a
                       AND np.token_b = tn.token_b

                    WHERE tn.entity_id = gt.source1_entity_id
                )

                OR

                EXISTS (
                    SELECT 1
                    FROM top2_address ta

                    WHERE
                        ta.entity_id = gt.source1_entity_id
                        AND ta.country = s3.country

                        AND EXISTS (
                            SELECT 1
                            FROM s3_address_token_index s3a
                            WHERE
                                s3a.entity_id = gt.matched_entity_id
                                AND s3a.token = ta.token1
                        )

                        AND EXISTS (
                            SELECT 1
                            FROM s3_address_token_index s3b
                            WHERE
                                s3b.entity_id = gt.matched_entity_id
                                AND s3b.token = ta.token2
                        )
                )

            THEN 1
            ELSE 0
        END
    ) AS captured_matches

FROM true_pairs gt

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s3_normalized s3
    ON gt.matched_entity_id = s3.entity_id
""").df()

combined_small_blocks_s3

,total_true_matches,captured_matches
0,3944746,3099754.0


In [46]:
total_s3_combined = int(
    combined_small_blocks_s3.loc[0, "total_true_matches"]
)

captured_s3_combined = int(
    combined_small_blocks_s3.loc[0, "captured_matches"]
)

print("S3 total true matches:", total_s3_combined)
print("S3 captured:", captured_s3_combined)

print(
    "S3 combined recall:",
    f"{captured_s3_combined / total_s3_combined:.4%}"
)

NameError: name 'combined_small_blocks_s3' is not defined

In [53]:
from pathlib import Path

TEST_DIR = DATASET_DIR / "test"

TEST_SOURCE1_PATH = (TEST_DIR / "test_source1.tsv").resolve()
TEST_SOURCE2_PATH = (TEST_DIR / "test_source2.tsv").resolve()
TEST_SOURCE3_PATH = (TEST_DIR / "test_source3.tsv").resolve()

con.execute(r"""
CREATE OR REPLACE TABLE test_source1_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM read_csv_auto(
    ?,
    delim='\t',
    header=true
)
""", [str(TEST_SOURCE1_PATH)])

con.execute(r"""
CREATE OR REPLACE TABLE test_source2_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM read_csv_auto(
    ?,
    delim='\t',
    header=true
)
""", [str(TEST_SOURCE2_PATH)])

con.execute(r"""
CREATE OR REPLACE TABLE test_source3_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM read_csv_auto(
    ?,
    delim='\t',
    header=true
)
""", [str(TEST_SOURCE3_PATH)])

print("Test normalized tables created.")

Test normalized tables created.


In [54]:
print(
    "Test S1:",
    con.execute("SELECT COUNT(*) FROM test_source1_normalized").fetchone()[0]
)

print(
    "Test S2:",
    con.execute("SELECT COUNT(*) FROM test_source2_normalized").fetchone()[0]
)

print(
    "Test S3:",
    con.execute("SELECT COUNT(*) FROM test_source3_normalized").fetchone()[0]
)

Test S1: 1732544
Test S2: 4887273
Test S3: 5082316


In [55]:
con.execute("""
CREATE OR REPLACE TABLE test_s2_name_frequency AS
SELECT
    normalized_name,
    COUNT(*) AS record_count
FROM test_source2_normalized
WHERE normalized_name != ''
GROUP BY normalized_name
""")

In [56]:
con.execute("""
CREATE OR REPLACE TABLE test_s3_name_frequency AS
SELECT
    normalized_name,
    COUNT(*) AS record_count
FROM test_source3_normalized
WHERE normalized_name != ''
GROUP BY normalized_name
""")

print("Test name-frequency tables created.")

Test name-frequency tables created.


In [57]:
con.execute("""
CREATE OR REPLACE TABLE test_s2_name_country_frequency AS
SELECT
    country,
    normalized_name,
    COUNT(*) AS record_count
FROM test_source2_normalized
WHERE normalized_name != ''
GROUP BY
    country,
    normalized_name
""")

con.execute("""
CREATE OR REPLACE TABLE test_s3_name_country_frequency AS
SELECT
    country,
    normalized_name,
    COUNT(*) AS record_count
FROM test_source3_normalized
WHERE normalized_name != ''
GROUP BY
    country,
    normalized_name
""")

print("Test country-specific name frequencies created.")

Test country-specific name frequencies created.


In [58]:
con.execute("""
CREATE OR REPLACE TABLE baseline_s2_name_matches AS

SELECT
    s1.entity_id AS source1_entity_id,
    s2.entity_id AS matched_entity_id

FROM test_source1_normalized s1

JOIN test_s2_name_country_frequency nf
    ON nf.country = s1.country
   AND nf.normalized_name = s1.normalized_name
   AND nf.record_count <= 3

JOIN test_source2_normalized s2
    ON s2.country = nf.country
   AND s2.normalized_name = nf.normalized_name

WHERE
    s1.normalized_name != ''
    AND nf.record_count > 0
""")

print("S2 exact-name matches created.")

S2 exact-name matches created.


In [59]:
con.execute("""
CREATE OR REPLACE TABLE baseline_s3_name_matches AS

SELECT
    s1.entity_id AS source1_entity_id,
    s3.entity_id AS matched_entity_id

FROM test_source1_normalized s1

JOIN test_s3_name_country_frequency nf
    ON nf.country = s1.country
   AND nf.normalized_name = s1.normalized_name
   AND nf.record_count <= 3

JOIN test_source3_normalized s3
    ON s3.country = nf.country
   AND s3.normalized_name = nf.normalized_name

WHERE
    s1.normalized_name != ''
    AND nf.record_count > 0
""")

print("S3 exact-name matches created.")

S3 exact-name matches created.


In [60]:
con.execute("""
CREATE OR REPLACE TABLE baseline_s2_address_matches AS

SELECT
    s1.entity_id AS source1_entity_id,
    s2.entity_id AS matched_entity_id

FROM test_source1_normalized s1

JOIN test_source2_normalized s2
    ON s1.country = s2.country
   AND s1.normalized_address = s2.normalized_address

WHERE
    s1.normalized_address != ''
    AND s2.normalized_address != ''
""")

print("S2 exact-address matches created.")

S2 exact-address matches created.


In [61]:
con.execute("""
CREATE OR REPLACE TABLE baseline_s3_address_matches AS

SELECT
    s1.entity_id AS source1_entity_id,
    s3.entity_id AS matched_entity_id

FROM test_source1_normalized s1

JOIN test_source3_normalized s3
    ON s1.country = s3.country
   AND s1.normalized_address = s3.normalized_address

WHERE
    s1.normalized_address != ''
    AND s3.normalized_address != ''
""")

print("S3 exact-address matches created.")

S3 exact-address matches created.


In [62]:
con.execute("""
CREATE OR REPLACE TABLE baseline_test_matches AS

SELECT DISTINCT
    source1_entity_id,
    matched_entity_id
FROM (
    SELECT * FROM baseline_s2_name_matches
    UNION ALL
    SELECT * FROM baseline_s3_name_matches
    UNION ALL
    SELECT * FROM baseline_s2_address_matches
    UNION ALL
    SELECT * FROM baseline_s3_address_matches
)
""")

print("Baseline test matches ready.")

Baseline test matches ready.


In [63]:
print(
    "Predicted pairs:",
    con.execute("""
        SELECT COUNT(*)
        FROM baseline_test_matches
    """).fetchone()[0]
)

print(
    "S1 entities with predictions:",
    con.execute("""
        SELECT COUNT(DISTINCT source1_entity_id)
        FROM baseline_test_matches
    """).fetchone()[0]
)

Predicted pairs: 2277869
S1 entities with predictions: 1100940


In [64]:
con.execute("""
CREATE OR REPLACE TABLE baseline_matching_results AS

SELECT
    s1.entity_id AS source1_entity_id,

    COALESCE(
        string_agg(
            DISTINCT m.matched_entity_id,
            ',' ORDER BY m.matched_entity_id
        ),
        ''
    ) AS matched_entity_ids

FROM test_source1_normalized s1

LEFT JOIN baseline_test_matches m
    ON s1.entity_id = m.source1_entity_id

GROUP BY
    s1.entity_id

ORDER BY
    s1.entity_id
""")

print("Matching results table created.")

Matching results table created.


In [65]:
con.execute("""
CREATE OR REPLACE TABLE baseline_candidate_results AS

SELECT
    s1.entity_id AS source1_entity_id,

    COALESCE(
        string_agg(
            DISTINCT m.matched_entity_id,
            ',' ORDER BY m.matched_entity_id
        ),
        ''
    ) AS candidate_entity_ids

FROM test_source1_normalized s1

LEFT JOIN baseline_test_matches m
    ON s1.entity_id = m.source1_entity_id

GROUP BY
    s1.entity_id

ORDER BY
    s1.entity_id
""")

print("Candidate results table created.")

Candidate results table created.


In [66]:
OUTPUT_DIR = PROJECT_ROOT / "output"

In [67]:
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

con.execute("""
COPY baseline_matching_results
TO 'output/matching_results.tsv'
WITH (
    HEADER TRUE,
    DELIMITER '\t'
)
""")

con.execute("""
COPY baseline_candidate_results
TO 'output/candidate_pairs.tsv'
WITH (
    HEADER TRUE,
    DELIMITER '\t'
)
""")

print("FIRST SUBMISSION FILES CREATED.")

FIRST SUBMISSION FILES CREATED.


In [68]:
print(
    (OUTPUT_DIR / "matching_results.tsv").resolve()
)

print(
    (OUTPUT_DIR / "candidate_pairs.tsv").resolve()
)

C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\notebooks\output\matching_results.tsv
C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\notebooks\output\candidate_pairs.tsv


In [69]:
con.execute("""
SELECT *
FROM read_csv(
    'output/matching_results.tsv',
    delim='\t',
    header=true
)
LIMIT 10
""").df()

,source1_entity_id,matched_entity_ids
0,S1-100000556,NaN
1,S1-100001013,S2-135727199
2,S1-100001295,"S2-334300764,S2-560501252,S3-367738979,S3-3802..."
3,S1-100001834,S2-900957143
4,S1-100003037,"S2-207887213,S3-906059588"
5,S1-100003550,NaN
6,S1-100003919,S2-373612556
7,S1-100004238,S2-236516835
8,S1-100005079,NaN
9,S1-100005491,NaN


In [70]:
from pathlib import Path
import csv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Project root: C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource
Output directory: C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\output


In [71]:
matching_path = OUTPUT_DIR / "matching_results.tsv"

cursor = con.execute("""
    SELECT
        source1_entity_id,
        matched_entity_ids
    FROM baseline_matching_results
    ORDER BY source1_entity_id
""")

with open(matching_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(
        f,
        delimiter="\t",
        lineterminator="\n",
        quoting=csv.QUOTE_MINIMAL
    )

    writer.writerow([
        "source1_entity_id",
        "matched_entity_ids"
    ])

    while True:
        rows = cursor.fetchmany(10000)

        if not rows:
            break

        for source1_id, matched_ids in rows:
            # Convert SQL NULL to a genuinely empty field
            matched_ids = "" if matched_ids is None else str(matched_ids)

            writer.writerow([
                source1_id,
                matched_ids
            ])

print("Written:", matching_path.resolve())

Written: C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\output\matching_results.tsv


In [72]:
candidate_path = OUTPUT_DIR / "candidate_pairs.tsv"

cursor = con.execute("""
    SELECT
        source1_entity_id,
        candidate_entity_ids
    FROM baseline_candidate_results
    ORDER BY source1_entity_id
""")

with open(candidate_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(
        f,
        delimiter="\t",
        lineterminator="\n",
        quoting=csv.QUOTE_MINIMAL
    )

    writer.writerow([
        "source1_entity_id",
        "candidate_entity_ids"
    ])

    while True:
        rows = cursor.fetchmany(10000)

        if not rows:
            break

        for source1_id, candidate_ids in rows:
            # Convert SQL NULL to a genuinely empty field
            candidate_ids = "" if candidate_ids is None else str(candidate_ids)

            writer.writerow([
                source1_id,
                candidate_ids
            ])

print("Written:", candidate_path.resolve())

Written: C:\Users\yugan\Desktop\6ab10eb3b23ba_student_resource\student_resource\output\candidate_pairs.tsv


In [73]:
with open(matching_path, "r", encoding="utf-8") as f:
    for _ in range(10):
        line = f.readline()

        print(repr(line))

'source1_entity_id\tmatched_entity_ids\n'
'S1-100000556\t\n'
'S1-100001013\tS2-135727199\n'
'S1-100001295\tS2-334300764,S2-560501252,S3-367738979,S3-380201812,S3-409832146\n'
'S1-100001834\tS2-900957143\n'
'S1-100003037\tS2-207887213,S3-906059588\n'
'S1-100003550\t\n'
'S1-100003919\tS2-373612556\n'
'S1-100004238\tS2-236516835\n'
'S1-100005079\t\n'


In [74]:
with open(candidate_path, "r", encoding="utf-8") as f:
    for _ in range(10):
        line = f.readline()

        print(repr(line))

'source1_entity_id\tcandidate_entity_ids\n'
'S1-100000556\t\n'
'S1-100001013\tS2-135727199\n'
'S1-100001295\tS2-334300764,S2-560501252,S3-367738979,S3-380201812,S3-409832146\n'
'S1-100001834\tS2-900957143\n'
'S1-100003037\tS2-207887213,S3-906059588\n'
'S1-100003550\t\n'
'S1-100003919\tS2-373612556\n'
'S1-100004238\tS2-236516835\n'
'S1-100005079\t\n'


In [75]:
prediction_stats = con.execute("""
WITH counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS match_count
    FROM baseline_test_matches
    GROUP BY source1_entity_id
)

SELECT
    COUNT(*) AS entities_with_predictions,
    SUM(CASE WHEN match_count = 1 THEN 1 ELSE 0 END) AS exactly_one,
    SUM(CASE WHEN match_count > 1 THEN 1 ELSE 0 END) AS multiple_matches,
    MAX(match_count) AS maximum_matches,
    AVG(match_count) AS average_matches
FROM counts
""").df()

prediction_stats

,entities_with_predictions,exactly_one,multiple_matches,maximum_matches,average_matches
0,1100940,487399.0,613541.0,129,2.069022


In [76]:
con.execute("""
SELECT
    match_count,
    COUNT(*) AS entity_count
FROM (
    SELECT
        source1_entity_id,
        COUNT(*) AS match_count
    FROM baseline_test_matches
    GROUP BY source1_entity_id
)
GROUP BY match_count
ORDER BY match_count
LIMIT 20
""").df()

,match_count,entity_count
0,1,487399
1,2,302617
2,3,180853
3,4,74254
4,5,32841
5,6,13971
6,7,3992
7,8,1467
8,9,878
9,10,568


In [77]:
con.execute("""
SELECT
    source1_entity_id,
    COUNT(*) AS match_count
FROM baseline_test_matches
GROUP BY source1_entity_id
ORDER BY match_count DESC
LIMIT 20
""").df()

,source1_entity_id,match_count
0,S1-300794653,129
1,S1-205949924,128
2,S1-167958929,127
3,S1-778561350,127
4,S1-202083314,127
5,S1-183373019,126
6,S1-370625971,126
7,S1-674712137,125
8,S1-237437156,125
9,S1-174277116,125


In [79]:
con.execute("""
CREATE OR REPLACE TABLE s2_fullname_country_frequency AS
SELECT
    country,
    normalized_name,
    COUNT(*) AS record_count
FROM test_source2_normalized
WHERE normalized_name != ''
GROUP BY
    country,
    normalized_name
""")

con.execute("""
CREATE OR REPLACE TABLE s3_fullname_country_frequency AS
SELECT
    country,
    normalized_name,
    COUNT(*) AS record_count
FROM test_source3_normalized
WHERE normalized_name != ''
GROUP BY
    country,
    normalized_name
""")

print("Training full-name frequency tables created.")

Training full-name frequency tables created.


In [80]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_s2_name AS
SELECT
    s1.entity_id AS source1_entity_id,
    s2.entity_id AS matched_entity_id

FROM test_source1_normalized s1

JOIN s2_fullname_country_frequency nf
    ON nf.country = s1.country
   AND nf.normalized_name = s1.normalized_name
   AND nf.record_count <= 3

JOIN test_source2_normalized s2
    ON s2.country = nf.country
   AND s2.normalized_name = nf.normalized_name

WHERE
    s1.normalized_name != ''
""")

print("S2 exact-name baseline pairs created.")

S2 exact-name baseline pairs created.


In [81]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_s3_name AS
SELECT
    s1.entity_id AS source1_entity_id,
    s3.entity_id AS matched_entity_id

FROM test_source1_normalized s1

JOIN s3_fullname_country_frequency nf
    ON nf.country = s1.country
   AND nf.normalized_name = s1.normalized_name
   AND nf.record_count <= 3

JOIN test_source3_normalized s3
    ON s3.country = nf.country
   AND s3.normalized_name = nf.normalized_name

WHERE
    s1.normalized_name != ''
""")

print("S3 exact-name baseline pairs created.")

S3 exact-name baseline pairs created.


In [83]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_s2_address AS
SELECT
    s1.entity_id AS source1_entity_id,
    s2.entity_id AS matched_entity_id

FROM test_source1_normalized s1

JOIN test_source2_normalized s2
    ON s1.country = s2.country
   AND s1.normalized_address = s2.normalized_address

WHERE
    s1.normalized_address != ''
    AND s2.normalized_address != ''
""")

print("S2 exact-address baseline pairs created.")

S2 exact-address baseline pairs created.


In [84]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_s3_address AS
SELECT
    s1.entity_id AS source1_entity_id,
    s3.entity_id AS matched_entity_id

FROM test_source1_normalized s1

JOIN test_source3_normalized s3
    ON s1.country = s3.country
   AND s1.normalized_address = s3.normalized_address

WHERE
    s1.normalized_address != ''
    AND s3.normalized_address != ''
""")

print("S3 exact-address baseline pairs created.")

S3 exact-address baseline pairs created.


In [85]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_pairs AS

SELECT DISTINCT
    source1_entity_id,
    matched_entity_id

FROM (
    SELECT * FROM train_baseline_s2_name
    UNION ALL
    SELECT * FROM train_baseline_s3_name
    UNION ALL
    SELECT * FROM train_baseline_s2_address
    UNION ALL
    SELECT * FROM train_baseline_s3_address
)
""")

print("Training baseline pairs combined.")

Training baseline pairs combined.


In [86]:
print(
    "Training baseline candidate pairs:",
    con.execute("""
        SELECT COUNT(*)
        FROM train_baseline_pairs
    """).fetchone()[0]
)

Training baseline candidate pairs: 2277869


In [88]:
baseline_train_score = con.execute("""
WITH predicted_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS predicted_count
    FROM train_baseline_pairs
    GROUP BY source1_entity_id
),

true_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS true_count
    FROM ground_truth_matches
    GROUP BY source1_entity_id
),

true_positive_counts AS (
    SELECT
        p.source1_entity_id,
        COUNT(*) AS tp
    FROM train_baseline_pairs p
    JOIN ground_truth_matches gt
        ON p.source1_entity_id = gt.source1_entity_id
       AND p.matched_entity_id = gt.matched_entity_id
    GROUP BY p.source1_entity_id
),

entity_scores AS (
    SELECT
        s1.entity_id,

        COALESCE(pc.predicted_count, 0) AS predicted_count,
        COALESCE(tc.true_count, 0) AS true_count,
        COALESCE(tp.tp, 0) AS tp

    FROM test_source1_normalized s1

    LEFT JOIN predicted_counts pc
        ON s1.entity_id = pc.source1_entity_id

    LEFT JOIN true_counts tc
        ON s1.entity_id = tc.source1_entity_id

    LEFT JOIN true_positive_counts tp
        ON s1.entity_id = tp.source1_entity_id
)

SELECT
    AVG(
        CASE
            WHEN predicted_count = 0
                 AND true_count = 0
            THEN 1.0

            WHEN predicted_count = 0
                 OR tp = 0
            THEN 0.0

            ELSE
                (
                    1.25
                    * (CAST(tp AS DOUBLE) / predicted_count)
                    * (CAST(tp AS DOUBLE) / true_count)
                )
                /
                (
                    0.25
                    * (CAST(tp AS DOUBLE) / predicted_count)
                    +
                    (CAST(tp AS DOUBLE) / true_count)
                )
        END
    ) AS macro_f05

FROM entity_scores
""").fetchone()[0]

print(
    f"Training baseline macro F0.5: {baseline_train_score:.6f}"
)

CatalogException: Catalog Error: Table with name ground_truth_matches does not exist!
Did you mean "duckdb_columns"?

LINE 14:     FROM ground_truth_matches
                  ^

In [92]:
con.execute(r"""
CREATE OR REPLACE TABLE ground_truth_matches AS

SELECT
    source1_entity_id,
    trim(matched_entity_id) AS matched_entity_id

FROM read_csv_auto(
    '../dataset/train/train_ground_truth.tsv',
    delim='\t',
    header=true
),

LATERAL unnest(
    string_split(
        matched_entity_ids,
        ','
    )
) AS t(matched_entity_id)

WHERE
    trim(matched_entity_id) != ''
""")

print("ground_truth_matches created.")

ground_truth_matches created.


In [93]:
con.execute("""
SELECT *
FROM ground_truth_matches
LIMIT 20
""").df()

,source1_entity_id,matched_entity_id
0,S1-287443814,S3-994590819
1,S1-532345566,S3-841106567
2,S1-218339847,S3-68132957
3,S1-754495308,S3-68334262
4,S1-245855772,S3-94368949
5,S1-754806114,S3-876122281
6,S1-968979856,S3-789397713
7,S1-276980785,S3-704675124
8,S1-663214807,S3-305445534
9,S1-741669017,S3-44637720


In [94]:
print(
    con.execute("""
        SELECT COUNT(*)
        FROM ground_truth_matches
    """).fetchone()[0]
)

7638365


In [95]:
print(
    "train_baseline_s2_name:",
    con.execute("""
        SELECT COUNT(*)
        FROM train_baseline_s2_name
    """).fetchone()[0]
)

print(
    "train_baseline_s3_name:",
    con.execute("""
        SELECT COUNT(*)
        FROM train_baseline_s3_name
    """).fetchone()[0]
)

print(
    "train_baseline_s2_address:",
    con.execute("""
        SELECT COUNT(*)
        FROM train_baseline_s2_address
    """).fetchone()[0]
)

print(
    "train_baseline_s3_address:",
    con.execute("""
        SELECT COUNT(*)
        FROM train_baseline_s3_address
    """).fetchone()[0]
)

train_baseline_s2_name: 901280
train_baseline_s3_name: 810437
train_baseline_s2_address: 391874
train_baseline_s3_address: 230089


In [96]:
con.execute("""
SELECT
    CASE
        WHEN matched_entity_id LIKE 'S2-%' THEN 'S2'
        WHEN matched_entity_id LIKE 'S3-%' THEN 'S3'
        ELSE 'OTHER'
    END AS source,
    COUNT(*) AS count
FROM ground_truth_matches
GROUP BY 1
ORDER BY 1
""").df()

,source,count
0,S2,3693619
1,S3,3944746


In [97]:
gt_raw = con.execute(r"""
SELECT
    COUNT(*) AS rows,
    COUNT(*) FILTER (
        WHERE matched_entity_ids IS NULL
           OR trim(matched_entity_ids) = ''
    ) AS empty_rows
FROM read_csv_auto(
    '../dataset/train/train_ground_truth.tsv',
    delim='\t',
    header=true
)
""").df()

gt_raw

,rows,empty_rows
0,2206821,123247


In [98]:
print(
    "Ground-truth rows:",
    gt_raw.loc[0, "rows"]
)

print(
    "Empty-match rows:",
    gt_raw.loc[0, "empty_rows"]
)

Ground-truth rows: 2206821
Empty-match rows: 123247


In [99]:
con.execute("""
SELECT
    matched_entity_id,
    COUNT(*) AS count
FROM ground_truth_matches
WHERE
    matched_entity_id NOT LIKE 'S2-%'
    AND matched_entity_id NOT LIKE 'S3-%'
GROUP BY matched_entity_id
ORDER BY count DESC
LIMIT 30
""").df()

,matched_entity_id,count


In [100]:
con.execute("""
SELECT
    COUNT(*) FILTER (
        WHERE matched_entity_id LIKE 'S2-%'
    ) AS s2_links,

    COUNT(*) FILTER (
        WHERE matched_entity_id LIKE 'S3-%'
    ) AS s3_links,

    COUNT(*) AS total_links
FROM ground_truth_matches
""").df()

,s2_links,s3_links,total_links
0,3693619,3944746,7638365


In [101]:
consistency_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE matched_entity_id LIKE 'S2-%'
    ) AS s2_rows,

    COUNT(*) FILTER (
        WHERE matched_entity_id LIKE 'S3-%'
    ) AS s3_rows,

    COUNT(*) FILTER (
        WHERE matched_entity_id NOT LIKE 'S2-%'
          AND matched_entity_id NOT LIKE 'S3-%'
    ) AS other_rows,

    COUNT(*) FILTER (
        WHERE matched_entity_id IS NULL
    ) AS null_rows

FROM ground_truth_matches
""").df()

consistency_check

,total_rows,s2_rows,s3_rows,other_rows,null_rows
0,7638365,3693619,3944746,0,0


In [102]:
consistency_check = con.execute("""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE matched_entity_id LIKE 'S2-%'
    ) AS s2_rows,

    COUNT(*) FILTER (
        WHERE matched_entity_id LIKE 'S3-%'
    ) AS s3_rows,

    COUNT(*) FILTER (
        WHERE matched_entity_id NOT LIKE 'S2-%'
          AND matched_entity_id NOT LIKE 'S3-%'
    ) AS other_rows,

    COUNT(*) FILTER (
        WHERE matched_entity_id IS NULL
    ) AS null_rows

FROM ground_truth_matches
""").df()

consistency_check

,total_rows,s2_rows,s3_rows,other_rows,null_rows
0,7638365,3693619,3944746,0,0


In [103]:
row = consistency_check.iloc[0]

print("S2 + S3 =", row["s2_rows"] + row["s3_rows"])
print("Reported total =", row["total_rows"])
print(
    "Difference =",
    row["total_rows"] - (row["s2_rows"] + row["s3_rows"])
)

S2 + S3 = 7638365
Reported total = 7638365
Difference = 0


In [104]:
duplicate_gt = con.execute("""
SELECT
    source1_entity_id,
    matched_entity_id,
    COUNT(*) AS duplicate_count
FROM ground_truth_matches
GROUP BY
    source1_entity_id,
    matched_entity_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20
""").df()

duplicate_gt

,source1_entity_id,matched_entity_id,duplicate_count


In [105]:
baseline_f05_result = con.execute("""
WITH
predicted_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS predicted_count
    FROM train_baseline_pairs
    GROUP BY source1_entity_id
),

true_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS true_count
    FROM ground_truth_matches
    GROUP BY source1_entity_id
),

true_positive_counts AS (
    SELECT
        p.source1_entity_id,
        COUNT(*) AS tp
    FROM train_baseline_pairs p

    INNER JOIN ground_truth_matches gt
        ON p.source1_entity_id = gt.source1_entity_id
       AND p.matched_entity_id = gt.matched_entity_id

    GROUP BY p.source1_entity_id
),

entity_scores AS (
    SELECT
        s1.entity_id,

        COALESCE(p.predicted_count, 0) AS predicted_count,
        COALESCE(t.true_count, 0) AS true_count,
        COALESCE(tp.tp, 0) AS tp

    FROM s1_normalized s1

    LEFT JOIN predicted_counts p
        ON s1.entity_id = p.source1_entity_id

    LEFT JOIN true_counts t
        ON s1.entity_id = t.source1_entity_id

    LEFT JOIN true_positive_counts tp
        ON s1.entity_id = tp.source1_entity_id
)

SELECT
    AVG(
        CASE

            /* Correct singleton prediction */
            WHEN predicted_count = 0
                 AND true_count = 0
            THEN 1.0

            /* Missed a real match OR false prediction on singleton */
            WHEN predicted_count = 0
                 OR true_count = 0
                 OR tp = 0
            THEN 0.0

            ELSE
                (
                    1.25
                    * (CAST(tp AS DOUBLE) / predicted_count)
                    * (CAST(tp AS DOUBLE) / true_count)
                )
                /
                (
                    0.25 * (CAST(tp AS DOUBLE) / predicted_count)
                    +
                    (CAST(tp AS DOUBLE) / true_count)
                )

        END
    ) AS macro_f05

FROM entity_scores
""").fetchone()[0]

print(
    f"Training baseline macro F0.5: {baseline_f05_result:.6f}"
)

CatalogException: Catalog Error: Table with name s1_normalized does not exist!
Did you mean "test_source1_normalized"?

LINE 40:     FROM s1_normalized s1
                  ^

In [106]:
con.execute(r"""
CREATE OR REPLACE TABLE s1_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM read_csv_auto(
    '../dataset/train/train_source1.tsv',
    delim='\t',
    header=true
)
""")

con.execute(r"""
CREATE OR REPLACE TABLE s2_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM read_csv_auto(
    '../dataset/train/train_source2.tsv',
    delim='\t',
    header=true
)
""")

con.execute(r"""
CREATE OR REPLACE TABLE s3_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM read_csv_auto(
    '../dataset/train/train_source3.tsv',
    delim='\t',
    header=true
)
""")

print("Training normalized tables recreated.")

Training normalized tables recreated.


In [107]:
print(
    "S1:",
    con.execute("SELECT COUNT(*) FROM s1_normalized").fetchone()[0]
)

print(
    "S2:",
    con.execute("SELECT COUNT(*) FROM s2_normalized").fetchone()[0]
)

print(
    "S3:",
    con.execute("SELECT COUNT(*) FROM s3_normalized").fetchone()[0]
)

S1: 2206821
S2: 5034616
S3: 5285603


In [108]:
baseline_f05_result = con.execute("""
WITH
predicted_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS predicted_count
    FROM train_baseline_pairs
    GROUP BY source1_entity_id
),

true_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS true_count
    FROM ground_truth_matches
    GROUP BY source1_entity_id
),

true_positive_counts AS (
    SELECT
        p.source1_entity_id,
        COUNT(*) AS tp
    FROM train_baseline_pairs p
    INNER JOIN ground_truth_matches gt
        ON p.source1_entity_id = gt.source1_entity_id
       AND p.matched_entity_id = gt.matched_entity_id
    GROUP BY p.source1_entity_id
),

entity_scores AS (
    SELECT
        s1.entity_id,
        COALESCE(p.predicted_count, 0) AS predicted_count,
        COALESCE(t.true_count, 0) AS true_count,
        COALESCE(tp.tp, 0) AS tp
    FROM s1_normalized s1
    LEFT JOIN predicted_counts p
        ON s1.entity_id = p.source1_entity_id
    LEFT JOIN true_counts t
        ON s1.entity_id = t.source1_entity_id
    LEFT JOIN true_positive_counts tp
        ON s1.entity_id = tp.source1_entity_id
)

SELECT
    AVG(
        CASE
            WHEN predicted_count = 0
                 AND true_count = 0
            THEN 1.0

            WHEN predicted_count = 0
                 OR true_count = 0
                 OR tp = 0
            THEN 0.0

            ELSE
                (
                    1.25
                    * (CAST(tp AS DOUBLE) / predicted_count)
                    * (CAST(tp AS DOUBLE) / true_count)
                )
                /
                (
                    0.25 * (CAST(tp AS DOUBLE) / predicted_count)
                    +
                    (CAST(tp AS DOUBLE) / true_count)
                )
        END
    ) AS macro_f05

FROM entity_scores
""").fetchone()[0]

print(f"Training baseline macro F0.5: {baseline_f05_result:.6f}")

Training baseline macro F0.5: 0.055848


In [109]:
baseline_diagnostic = con.execute("""
WITH
predicted_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS predicted_count
    FROM train_baseline_pairs
    GROUP BY source1_entity_id
),

true_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS true_count
    FROM ground_truth_matches
    GROUP BY source1_entity_id
),

tp_counts AS (
    SELECT
        p.source1_entity_id,
        COUNT(*) AS tp
    FROM train_baseline_pairs p
    INNER JOIN ground_truth_matches gt
        ON p.source1_entity_id = gt.source1_entity_id
       AND p.matched_entity_id = gt.matched_entity_id
    GROUP BY p.source1_entity_id
),

entity_stats AS (
    SELECT
        s1.entity_id,

        COALESCE(p.predicted_count, 0) AS predicted_count,
        COALESCE(t.true_count, 0) AS true_count,
        COALESCE(tp.tp, 0) AS tp

    FROM s1_normalized s1

    LEFT JOIN predicted_counts p
        ON s1.entity_id = p.source1_entity_id

    LEFT JOIN true_counts t
        ON s1.entity_id = t.source1_entity_id

    LEFT JOIN tp_counts tp
        ON s1.entity_id = tp.source1_entity_id
)

SELECT

    COUNT(*) AS total_s1,

    SUM(
        CASE WHEN true_count > 0 THEN 1 ELSE 0 END
    ) AS non_singletons,

    SUM(
        CASE WHEN true_count = 0 THEN 1 ELSE 0 END
    ) AS singletons,

    SUM(
        CASE
            WHEN true_count = 0
             AND predicted_count = 0
            THEN 1
            ELSE 0
        END
    ) AS correct_empty_singletons,

    SUM(
        CASE WHEN predicted_count > 0 THEN 1 ELSE 0 END
    ) AS entities_with_predictions,

    SUM(tp) AS total_true_positives,

    SUM(predicted_count) AS total_predictions,

    SUM(true_count) AS total_true_links

FROM entity_stats
""").df()

baseline_diagnostic

,total_s1,non_singletons,singletons,correct_empty_singletons,entities_with_predictions,total_true_positives,total_predictions,total_true_links
0,2206821,2083574.0,123247.0,123247.0,0.0,0.0,0.0,7638365.0


In [110]:
d = baseline_diagnostic.iloc[0]

print("Total S1:", int(d["total_s1"]))
print("Non-singletons:", int(d["non_singletons"]))
print("Singletons:", int(d["singletons"]))
print("Correct empty singletons:", int(d["correct_empty_singletons"]))
print("Entities with predictions:", int(d["entities_with_predictions"]))
print("Total predicted pairs:", int(d["total_predictions"]))
print("Total true positives:", int(d["total_true_positives"]))
print("Total true links:", int(d["total_true_links"]))

print(
    "Pair precision:",
    f"{d['total_true_positives'] / d['total_predictions']:.4%}"
)

print(
    "Pair recall:",
    f"{d['total_true_positives'] / d['total_true_links']:.4%}"
)

Total S1: 2206821
Non-singletons: 2083574
Singletons: 123247
Correct empty singletons: 123247
Entities with predictions: 0
Total predicted pairs: 0
Total true positives: 0
Total true links: 7638365
Pair precision: nan%
Pair recall: 0.0000%


C:\Users\yugan\AppData\Local\Temp\ipykernel_23036\3061924712.py:14: RuntimeWarning: invalid value encountered in scalar divide
  f"{d['total_true_positives'] / d['total_predictions']:.4%}"


In [111]:
con.execute("""
WITH
predicted AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS predicted_count
    FROM train_baseline_pairs
    GROUP BY source1_entity_id
),

tp AS (
    SELECT
        p.source1_entity_id,
        COUNT(*) AS tp
    FROM train_baseline_pairs p
    JOIN ground_truth_matches gt
        ON p.source1_entity_id = gt.source1_entity_id
       AND p.matched_entity_id = gt.matched_entity_id
    GROUP BY p.source1_entity_id
)

SELECT
    SUM(
        CASE
            WHEN COALESCE(tp.tp, 0) > 0
            THEN 1
            ELSE 0
        END
    ) AS entities_with_at_least_one_tp,

    SUM(
        CASE
            WHEN COALESCE(tp.tp, 0) = 0
             AND COALESCE(predicted.predicted_count, 0) > 0
            THEN 1
            ELSE 0
        END
    ) AS entities_with_only_false_predictions

FROM s1_normalized s1

LEFT JOIN predicted
    ON s1.entity_id = predicted.source1_entity_id

LEFT JOIN tp
    ON s1.entity_id = tp.source1_entity_id
""").df()

,entities_with_at_least_one_tp,entities_with_only_false_predictions
0,0.0,0.0


In [112]:
import duckdb

DB_PATH = "../er_work.duckdb"

con = duckdb.connect(DB_PATH)

print("Persistent DuckDB connected:", DB_PATH)

Persistent DuckDB connected: ../er_work.duckdb


In [113]:
print(
    con.execute("SELECT current_database()").fetchone()[0]
)

er_work


In [114]:
con.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name
""").df()

,table_name


In [115]:
con.execute(r"""
CREATE OR REPLACE TABLE s1_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM read_csv_auto(
    '../dataset/train/train_source1.tsv',
    delim='\t',
    header=true
)
""")

con.execute(r"""
CREATE OR REPLACE TABLE s2_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM read_csv_auto(
    '../dataset/train/train_source2.tsv',
    delim='\t',
    header=true
)
""")

con.execute(r"""
CREATE OR REPLACE TABLE s3_normalized AS
SELECT
    entity_id,
    business_name,
    business_address,
    country,

    regexp_replace(
        regexp_replace(
            lower(business_name),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_name,

    regexp_replace(
        regexp_replace(
            lower(business_address),
            '[^\p{L}\p{M}\p{N}\s]',
            ' ',
            'g'
        ),
        '\s+',
        ' ',
        'g'
    ) AS normalized_address

FROM read_csv_auto(
    '../dataset/train/train_source3.tsv',
    delim='\t',
    header=true
)
""")

print("Training normalized tables saved to persistent DB.")

Training normalized tables saved to persistent DB.


In [116]:
con.execute(r"""
CREATE OR REPLACE TABLE ground_truth_matches AS

SELECT
    source1_entity_id,
    trim(matched_entity_id) AS matched_entity_id

FROM read_csv_auto(
    '../dataset/train/train_ground_truth.tsv',
    delim='\t',
    header=true
),

LATERAL unnest(
    string_split(
        matched_entity_ids,
        ','
    )
) AS t(matched_entity_id)

WHERE trim(matched_entity_id) != ''
""")

print("Ground truth saved.")

Ground truth saved.


In [117]:
print(
    "S1:",
    con.execute("SELECT COUNT(*) FROM s1_normalized").fetchone()[0]
)

print(
    "S2:",
    con.execute("SELECT COUNT(*) FROM s2_normalized").fetchone()[0]
)

print(
    "S3:",
    con.execute("SELECT COUNT(*) FROM s3_normalized").fetchone()[0]
)

print(
    "Ground-truth links:",
    con.execute("SELECT COUNT(*) FROM ground_truth_matches").fetchone()[0]
)

S1: 2206821
S2: 5034616
S3: 5285603
Ground-truth links: 7638365


In [118]:
print(
    "train_baseline_pairs rows:",
    con.execute("""
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_name = 'train_baseline_pairs'
    """).fetchone()[0]
)

train_baseline_pairs rows: 0


In [119]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_s2_name AS
SELECT
    s1.entity_id AS source1_entity_id,
    s2.entity_id AS matched_entity_id

FROM s1_normalized s1

JOIN (
    SELECT
        country,
        normalized_name
    FROM s2_normalized
    WHERE normalized_name != ''
    GROUP BY country, normalized_name
    HAVING COUNT(*) <= 3
) n
    ON n.country = s1.country
   AND n.normalized_name = s1.normalized_name

JOIN s2_normalized s2
    ON s2.country = n.country
   AND s2.normalized_name = n.normalized_name

WHERE s1.normalized_name != ''
""")

print("S2 name block:", 
      con.execute("SELECT COUNT(*) FROM train_baseline_s2_name").fetchone()[0])

S2 name block: 1095328


In [120]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_s3_name AS
SELECT
    s1.entity_id AS source1_entity_id,
    s3.entity_id AS matched_entity_id

FROM s1_normalized s1

JOIN (
    SELECT
        country,
        normalized_name
    FROM s3_normalized
    WHERE normalized_name != ''
    GROUP BY country, normalized_name
    HAVING COUNT(*) <= 3
) n
    ON n.country = s1.country
   AND n.normalized_name = s1.normalized_name

JOIN s3_normalized s3
    ON s3.country = n.country
   AND s3.normalized_name = n.normalized_name

WHERE s1.normalized_name != ''
""")

print("S3 name block:",
      con.execute("SELECT COUNT(*) FROM train_baseline_s3_name").fetchone()[0])

S3 name block: 1014522


In [121]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_s2_address AS
SELECT
    s1.entity_id AS source1_entity_id,
    s2.entity_id AS matched_entity_id

FROM s1_normalized s1

JOIN s2_normalized s2
    ON s1.country = s2.country
   AND s1.normalized_address = s2.normalized_address

WHERE
    s1.normalized_address != ''
    AND s2.normalized_address != ''
""")

print("S2 address block:",
      con.execute("SELECT COUNT(*) FROM train_baseline_s2_address").fetchone()[0])

S2 address block: 527396


In [122]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_s3_address AS
SELECT
    s1.entity_id AS source1_entity_id,
    s3.entity_id AS matched_entity_id

FROM s1_normalized s1

JOIN s3_normalized s3
    ON s1.country = s3.country
   AND s1.normalized_address = s3.normalized_address

WHERE
    s1.normalized_address != ''
    AND s3.normalized_address != ''
""")

print("S3 address block:",
      con.execute("SELECT COUNT(*) FROM train_baseline_s3_address").fetchone()[0])

S3 address block: 205504


In [123]:
con.execute("""
CREATE OR REPLACE TABLE train_baseline_pairs AS

SELECT DISTINCT
    source1_entity_id,
    matched_entity_id

FROM (
    SELECT * FROM train_baseline_s2_name
    UNION ALL
    SELECT * FROM train_baseline_s3_name
    UNION ALL
    SELECT * FROM train_baseline_s2_address
    UNION ALL
    SELECT * FROM train_baseline_s3_address
)
""")

train_pairs_count = con.execute("""
SELECT COUNT(*)
FROM train_baseline_pairs
""").fetchone()[0]

print("Unique training baseline pairs:", train_pairs_count)

Unique training baseline pairs: 2780372


In [124]:
con.execute("""
SELECT
    COUNT(*) AS baseline_pairs,
    COUNT(gt.matched_entity_id) AS true_positive_rows

FROM train_baseline_pairs p

LEFT JOIN ground_truth_matches gt
    ON p.source1_entity_id = gt.source1_entity_id
   AND p.matched_entity_id = gt.matched_entity_id
""").df()

,baseline_pairs,true_positive_rows
0,2780372,1646712


In [125]:
baseline_f05_result = con.execute("""
WITH predicted_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS predicted_count
    FROM train_baseline_pairs
    GROUP BY source1_entity_id
),

true_counts AS (
    SELECT
        source1_entity_id,
        COUNT(*) AS true_count
    FROM ground_truth_matches
    GROUP BY source1_entity_id
),

true_positive_counts AS (
    SELECT
        p.source1_entity_id,
        COUNT(*) AS tp
    FROM train_baseline_pairs p
    INNER JOIN ground_truth_matches gt
        ON p.source1_entity_id = gt.source1_entity_id
       AND p.matched_entity_id = gt.matched_entity_id
    GROUP BY p.source1_entity_id
),

entity_scores AS (
    SELECT
        s1.entity_id,
        COALESCE(p.predicted_count, 0) AS predicted_count,
        COALESCE(t.true_count, 0) AS true_count,
        COALESCE(tp.tp, 0) AS tp
    FROM s1_normalized s1

    LEFT JOIN predicted_counts p
        ON s1.entity_id = p.source1_entity_id

    LEFT JOIN true_counts t
        ON s1.entity_id = t.source1_entity_id

    LEFT JOIN true_positive_counts tp
        ON s1.entity_id = tp.source1_entity_id
)

SELECT
    AVG(
        CASE
            WHEN predicted_count = 0
                 AND true_count = 0
                THEN 1.0

            WHEN predicted_count = 0
                 OR true_count = 0
                 OR tp = 0
                THEN 0.0

            ELSE
                (
                    1.25
                    * (CAST(tp AS DOUBLE) / predicted_count)
                    * (CAST(tp AS DOUBLE) / true_count)
                )
                /
                (
                    0.25 * (CAST(tp AS DOUBLE) / predicted_count)
                    +
                    (CAST(tp AS DOUBLE) / true_count)
                )
        END
    ) AS macro_f05

FROM entity_scores
""").fetchone()[0]

print(f"Training baseline macro F0.5: {baseline_f05_result:.6f}")

Training baseline macro F0.5: 0.370570


In [126]:
con.execute("""
CREATE OR REPLACE TABLE missed_true_pairs AS

SELECT
    gt.source1_entity_id,
    gt.matched_entity_id

FROM ground_truth_matches gt

WHERE NOT EXISTS (
    SELECT 1
    FROM train_baseline_pairs p

    WHERE
        p.source1_entity_id = gt.source1_entity_id
        AND p.matched_entity_id = gt.matched_entity_id
)
""")

print(
    "Missed true pairs:",
    con.execute("""
        SELECT COUNT(*)
        FROM missed_true_pairs
    """).fetchone()[0]
)

Missed true pairs: 5991653


In [127]:
con.execute("""
SELECT
    CASE
        WHEN matched_entity_id LIKE 'S2-%' THEN 'S2'
        WHEN matched_entity_id LIKE 'S3-%' THEN 'S3'
    END AS source,
    COUNT(*) AS missed_pairs
FROM missed_true_pairs
GROUP BY 1
ORDER BY 1
""").df()

,source,missed_pairs
0,S2,2791899
1,S3,3199754


In [128]:
miss_reason_summary = con.execute("""
WITH missed AS (
    SELECT
        m.source1_entity_id,
        m.matched_entity_id,

        s1.normalized_name AS s1_name,
        s1.normalized_address AS s1_address,

        CASE
            WHEN m.matched_entity_id LIKE 'S2-%'
            THEN s2.normalized_name
            ELSE s3.normalized_name
        END AS target_name,

        CASE
            WHEN m.matched_entity_id LIKE 'S2-%'
            THEN s2.normalized_address
            ELSE s3.normalized_address
        END AS target_address

    FROM missed_true_pairs m

    JOIN s1_normalized s1
        ON m.source1_entity_id = s1.entity_id

    LEFT JOIN s2_normalized s2
        ON m.matched_entity_id = s2.entity_id

    LEFT JOIN s3_normalized s3
        ON m.matched_entity_id = s3.entity_id
)

SELECT
    COUNT(*) AS missed_total,

    SUM(
        CASE
            WHEN s1_name != ''
             AND s1_name = target_name
            THEN 1 ELSE 0
        END
    ) AS exact_name_still_missed,

    SUM(
        CASE
            WHEN s1_address != ''
             AND s1_address = target_address
            THEN 1 ELSE 0
        END
    ) AS exact_address_still_missed,

    SUM(
        CASE
            WHEN s1_name = ''
              OR target_name = ''
            THEN 1 ELSE 0
        END
    ) AS missing_name,

    SUM(
        CASE
            WHEN s1_address = ''
              OR target_address = ''
            THEN 1 ELSE 0
        END
    ) AS missing_address

FROM missed
""").df()

miss_reason_summary

,missed_total,exact_name_still_missed,exact_address_still_missed,missing_name,missing_address
0,5991653,277959.0,0.0,0.0,0.0


In [129]:
con.execute("""
SELECT
    m.source1_entity_id,
    m.matched_entity_id,

    s1.business_name AS source1_name,
    s2.business_name AS s2_name,
    s3.business_name AS s3_name,

    s1.business_address AS source1_address,
    s2.business_address AS s2_address,
    s3.business_address AS s3_address,

    s1.country

FROM missed_true_pairs m

JOIN s1_normalized s1
    ON m.source1_entity_id = s1.entity_id

LEFT JOIN s2_normalized s2
    ON m.matched_entity_id = s2.entity_id

LEFT JOIN s3_normalized s3
    ON m.matched_entity_id = s3.entity_id

LIMIT 100
""").df()

,source1_entity_id,matched_entity_id,source1_name,s2_name,s3_name,source1_address,s2_address,s3_address,country
0,S1-350837867,S2-875538830,Modern Power Limited,मॉडर्न पावर लिमिटेड,None,"H.No. 140 - B, Hariganga Nagar, Hoshangabad Ro...","H.NO. 140/3 - B, BHOPAL, Madhya Pradesh",None,India
1,S1-409021324,S2-15307006,Surgical Clinic LP,Surgical Clinic LP,None,"79 Lexington Court, Pleasant Hill, IA","79 LEXINGTON CT, PLEASANT HILL, IA",None,US
2,S1-364278008,S2-203243568,American Program Inc,... americanprogram.com,None,"102 Scarborough Place, VA, Albemarle County","102. SCARBOROUGH PL, NULL, CHARLOTTESVILLE, VA",None,US
3,S1-769941577,S2-155546430,Gip & Brothers Private Limited,Gip & Brothers,None,"Office No.201, 9930 Space, Building, Mangal Na...","OFFICE NO.B3/201, 9930 SPACE, BUILDING, MAGAL ...",None,India
4,S1-360067729,S2-184525252,Raigad (India) Net Pvt. Ltd.,Raigad-(India) Net,None,"C-7/ 270C, Lawrence Road, Keshav Puram, Delhi,...","C-##7/ 270C, LAWRENCE ROAD, KESHAV PURAM, DELH...",None,India
...,...,...,...,...,...,...,...,...,...
95,S1-567223962,S2-787710244,Jawahar Systems Private Limited,Jawahar 5ystems Private Limited,None,"4-31, Puretivari Veedhi, Savalyapuram, Guntur,...","DOOR NO 805 4-31, PURETIVARI VEEDHI, SAVALYAPU...",None,India
96,S1-585311564,S2-478846783,National Cascade Stifel PLLC,NATIONAL CASCADE STIFEL (PLLC),None,"3365 Foster Drive, Baton Rouge City, LA","3365 FOSTER DRIVE, BATON ROUGE, LA",None,US
97,S1-304804127,S2-506711391,Flores Express Elm,FLORES EXPRESS,None,"16 Jackson Avenue, Batavia, NY","16 JACNSON AVENUE, BATAVIA, NY",None,US
98,S1-171970067,S2-358588516,"Jefferson and Miller, LLC","JEFFERSON & MILLNE,R LLC",None,"4321 Maypole Avenue, Chicago, IL","IL, MAYPOLE AVENUE, CHICAGOT OWNSHIP",None,US


In [130]:
con.execute("""
CREATE OR REPLACE TABLE s1_fullname_country_frequency AS
SELECT
    country,
    normalized_name,
    COUNT(*) AS record_count
FROM s1_normalized
WHERE normalized_name != ''
GROUP BY
    country,
    normalized_name
""")

print("S1 full-name frequencies created.")

S1 full-name frequencies created.


In [131]:
con.execute("""
CREATE OR REPLACE TABLE s2_fullname_country_frequency AS
SELECT
    country,
    normalized_name,
    COUNT(*) AS record_count
FROM s2_normalized
WHERE normalized_name != ''
GROUP BY
    country,
    normalized_name
""")

print("S2 full-name frequencies created.")

S2 full-name frequencies created.


In [132]:
con.execute("""
CREATE OR REPLACE TABLE s3_fullname_country_frequency AS
SELECT
    country,
    normalized_name,
    COUNT(*) AS record_count
FROM s3_normalized
WHERE normalized_name != ''
GROUP BY
    country,
    normalized_name
""")

print("S3 full-name frequencies created.")

S3 full-name frequencies created.


In [133]:
name_caps = [3, 5, 10, 25, 50, 100, 250, 500, 1000, 5000]

In [134]:
s2_name_threshold_results = []

for cap in name_caps:

    candidate_pairs = con.execute(f"""
        SELECT
            COALESCE(
                SUM(s1.record_count * s2.record_count),
                0
            )
        FROM s1_fullname_country_frequency s1

        JOIN s2_fullname_country_frequency s2
            ON s1.country = s2.country
           AND s1.normalized_name = s2.normalized_name

        WHERE s2.record_count <= {cap}
    """).fetchone()[0]

    total_matches, captured = con.execute(f"""
        SELECT
            COUNT(*),

            COUNT(*) FILTER (
                WHERE nf.record_count <= {cap}
            )

        FROM ground_truth_matches gt

        JOIN s2_normalized s2
            ON gt.matched_entity_id = s2.entity_id

        JOIN s2_fullname_country_frequency nf
            ON s2.country = nf.country
           AND s2.normalized_name = nf.normalized_name

        WHERE gt.matched_entity_id LIKE 'S2-%'
    """).fetchone()

    s2_name_threshold_results.append({
        "cap": cap,
        "candidate_pairs": int(candidate_pairs),
        "avg_candidates_per_s1": float(candidate_pairs / 2206821),
        "recall": float(captured / total_matches)
    })

pd.DataFrame(s2_name_threshold_results)

,cap,candidate_pairs,avg_candidates_per_s1,recall
0,3,1095328,0.496337,0.847701
1,5,1508344,0.683492,0.888323
2,10,1960313,0.888297,0.927075
3,25,3130808,1.418696,0.964260
4,50,5298126,2.400796,0.987222
5,100,7559822,3.425662,0.995135
6,250,8608025,3.900645,0.998485
7,500,8608506,3.900863,1.000000
8,1000,8608506,3.900863,1.000000
9,5000,8608506,3.900863,1.000000


In [135]:
s2_exact_name_all_candidates = con.execute("""
SELECT
    COALESCE(
        SUM(s1.record_count * s2.record_count),
        0
    )
FROM s1_fullname_country_frequency s1

JOIN s2_fullname_country_frequency s2
    ON s1.country = s2.country
   AND s1.normalized_name = s2.normalized_name
""").fetchone()[0]

print(
    "S2 unrestricted exact-name candidates:",
    int(s2_exact_name_all_candidates)
)

print(
    "Average per S1:",
    s2_exact_name_all_candidates / 2206821
)

S2 unrestricted exact-name candidates: 8608506
Average per S1: 3.9008628248507695


In [136]:
s2_exact_name_all_recall = con.execute("""
SELECT
    COUNT(*) AS total_matches,

    COUNT(*) FILTER (
        WHERE
            s2.normalized_name != ''
            AND s1.normalized_name = s2.normalized_name
    ) AS captured

FROM ground_truth_matches gt

JOIN s1_normalized s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_normalized s2
    ON gt.matched_entity_id = s2.entity_id

WHERE gt.matched_entity_id LIKE 'S2-%'
""").df()

s2_exact_name_all_recall["recall"] = (
    s2_exact_name_all_recall["captured"]
    / s2_exact_name_all_recall["total_matches"]
)

s2_exact_name_all_recall

,total_matches,captured,recall
0,3693619,665725,0.180237


In [137]:
s2_name_threshold_results_corrected = []

for cap in name_caps:

    candidate_pairs = con.execute(f"""
        SELECT
            COALESCE(
                SUM(s1.record_count * s2.record_count),
                0
            )
        FROM s1_fullname_country_frequency s1

        JOIN s2_fullname_country_frequency s2
            ON s1.country = s2.country
           AND s1.normalized_name = s2.normalized_name

        WHERE s1.normalized_name != ''
          AND s2.normalized_name != ''
          AND s2.record_count <= {cap}
    """).fetchone()[0]

    total_matches, captured = con.execute(f"""
        SELECT
            COUNT(*) AS total_matches,

            COUNT(*) FILTER (
                WHERE
                    s1.normalized_name != ''
                    AND s2.normalized_name != ''
                    AND s1.normalized_name = s2.normalized_name
                    AND nf.record_count <= {cap}
            ) AS captured

        FROM ground_truth_matches gt

        JOIN s1_normalized s1
            ON gt.source1_entity_id = s1.entity_id

        JOIN s2_normalized s2
            ON gt.matched_entity_id = s2.entity_id

        JOIN s2_fullname_country_frequency nf
            ON s2.country = nf.country
           AND s2.normalized_name = nf.normalized_name

        WHERE gt.matched_entity_id LIKE 'S2-%'
    """).fetchone()

    s2_name_threshold_results_corrected.append({
        "cap": cap,
        "candidate_pairs": int(candidate_pairs),
        "avg_candidates_per_s1": candidate_pairs / 2206821,
        "recall": captured / total_matches
    })

pd.DataFrame(s2_name_threshold_results_corrected)

,cap,candidate_pairs,avg_candidates_per_s1,recall
0,3,1095328,0.496337,0.144294
1,5,1508344,0.683492,0.152587
2,10,1960313,0.888297,0.158690
3,25,3130808,1.418696,0.166212
4,50,5298126,2.400796,0.174561
5,100,7559822,3.425662,0.178816
6,250,8608025,3.900645,0.180236
7,500,8608506,3.900863,0.180237
8,1000,8608506,3.900863,0.180237
9,5000,8608506,3.900863,0.180237


In [138]:
s3_name_threshold_results_corrected = []

for cap in name_caps:

    candidate_pairs = con.execute(f"""
        SELECT
            COALESCE(
                SUM(s1.record_count * s3.record_count),
                0
            )
        FROM s1_fullname_country_frequency s1

        JOIN s3_fullname_country_frequency s3
            ON s1.country = s3.country
           AND s1.normalized_name = s3.normalized_name

        WHERE s1.normalized_name != ''
          AND s3.normalized_name != ''
          AND s3.record_count <= {cap}
    """).fetchone()[0]

    total_matches, captured = con.execute(f"""
        SELECT
            COUNT(*) AS total_matches,

            COUNT(*) FILTER (
                WHERE
                    s1.normalized_name != ''
                    AND s3.normalized_name != ''
                    AND s1.normalized_name = s3.normalized_name
                    AND nf.record_count <= {cap}
            ) AS captured

        FROM ground_truth_matches gt

        JOIN s1_normalized s1
            ON gt.source1_entity_id = s1.entity_id

        JOIN s3_normalized s3
            ON gt.matched_entity_id = s3.entity_id

        JOIN s3_fullname_country_frequency nf
            ON s3.country = nf.country
           AND s3.normalized_name = nf.normalized_name

        WHERE gt.matched_entity_id LIKE 'S3-%'
    """).fetchone()

    s3_name_threshold_results_corrected.append({
        "cap": cap,
        "candidate_pairs": int(candidate_pairs),
        "avg_candidates_per_s1": candidate_pairs / 2206821,
        "recall": captured / total_matches
    })

pd.DataFrame(s3_name_threshold_results_corrected)

,cap,candidate_pairs,avg_candidates_per_s1,recall
0,3,1014522,0.459721,0.145691
1,5,1427126,0.646689,0.154465
2,10,2495930,1.131007,0.164440
3,25,3828333,1.734773,0.172714
4,50,6023988,2.729713,0.180868
5,100,8210839,3.720664,0.184948
6,250,9472680,4.292455,0.186620
7,500,9473160,4.292673,0.186620
8,1000,9473160,4.292673,0.186620
9,5000,9473160,4.292673,0.186620


In [139]:
con.execute("""
CREATE OR REPLACE TABLE s1_core_name AS

SELECT
    entity_id,
    country,
    normalized_name,

    regexp_replace(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    normalized_name,

                    '\\b(private|pvt)\\b',
                    ' ',
                    'g'
                ),

                '\\b(limited|ltd)\\b',
                ' ',
                'g'
            ),

            '\\b(incorporated|inc)\\b',
            ' ',
            'g'
        ),

        '\\b(corporation|corp)\\b',
        ' ',
        'g'
    ) AS core_name

FROM s1_normalized
""")

con.execute("""
CREATE OR REPLACE TABLE s2_core_name AS

SELECT
    entity_id,
    country,
    normalized_name,

    regexp_replace(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    normalized_name,

                    '\\b(private|pvt)\\b',
                    ' ',
                    'g'
                ),

                '\\b(limited|ltd)\\b',
                ' ',
                'g'
            ),

            '\\b(incorporated|inc)\\b',
            ' ',
            'g'
        ),

        '\\b(corporation|corp)\\b',
        ' ',
        'g'
    ) AS core_name

FROM s2_normalized
""")

con.execute("""
CREATE OR REPLACE TABLE s3_core_name AS

SELECT
    entity_id,
    country,
    normalized_name,

    regexp_replace(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    normalized_name,

                    '\\b(private|pvt)\\b',
                    ' ',
                    'g'
                ),

                '\\b(limited|ltd)\\b',
                ' ',
                'g'
            ),

            '\\b(incorporated|inc)\\b',
            ' ',
            'g'
        ),

        '\\b(corporation|corp)\\b',
        ' ',
        'g'
    ) AS core_name

FROM s3_normalized
""")

print("Core-name tables created.")

Core-name tables created.


In [140]:
for table in ["s1_core_name", "s2_core_name", "s3_core_name"]:
    con.execute(f"""
        UPDATE {table}
        SET core_name = regexp_replace(
            trim(core_name),
            '\\s+',
            ' ',
            'g'
        )
    """)

print("Core names cleaned.")

Core names cleaned.


In [141]:
con.execute("""
SELECT
    normalized_name,
    core_name
FROM s2_core_name
WHERE normalized_name != ''
LIMIT 50
""").df()

,normalized_name,core_name
0,राम मार्केटिंग प्राइवेट लिमिटेड,राम मार्केटिंग प्राइवेट लिमिटेड
1,holloway peak inc seafood,holloway peak seafood
2,आदित्य प्रॉपर्टीज एलएलपी,आदित्य प्रॉपर्टीज एलएलपी
3,summit inc,summit
4,delta tetlecommunication inc,delta tetlecommunication
5,lee and lawson,lee and lawson
6,shivshakti vidyalaya vidyalaya overseas corpor...,shivshakti vidyalaya vidyalaya overseas www sh...
7,shree infracon private ltd,shree infracon
8,chavira platinum chimera llc,chavira platinum chimera llc
9,foundation excel agency private limited,foundation excel agency


In [142]:
s2_core_name_recall = con.execute("""
SELECT
    COUNT(*) AS total_matches,

    COUNT(*) FILTER (
        WHERE
            s1.core_name != ''
            AND s2.core_name != ''
            AND s1.core_name = s2.core_name
            AND s1.country = s2.country
    ) AS captured

FROM ground_truth_matches gt

JOIN s1_core_name s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_core_name s2
    ON gt.matched_entity_id = s2.entity_id

WHERE gt.matched_entity_id LIKE 'S2-%'
""").df()

s2_core_name_recall["recall"] = (
    s2_core_name_recall["captured"]
    / s2_core_name_recall["total_matches"]
)

s2_core_name_recall

,total_matches,captured,recall
0,3693619,1350369,0.365595


In [143]:
s3_core_name_recall = con.execute("""
SELECT
    COUNT(*) AS total_matches,

    COUNT(*) FILTER (
        WHERE
            s1.core_name != ''
            AND s3.core_name != ''
            AND s1.core_name = s3.core_name
            AND s1.country = s3.country
    ) AS captured

FROM ground_truth_matches gt

JOIN s1_core_name s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s3_core_name s3
    ON gt.matched_entity_id = s3.entity_id

WHERE gt.matched_entity_id LIKE 'S3-%'
""").df()

s3_core_name_recall["recall"] = (
    s3_core_name_recall["captured"]
    / s3_core_name_recall["total_matches"]
)

s3_core_name_recall

,total_matches,captured,recall
0,3944746,1416961,0.359202


In [144]:
LEGAL_FORMS_V2 = (
    "private|pvt|प्राइवेट|"
    "limited|ltd|लिमिटेड|"
    "incorporated|inc|इंक|"
    "corporation|corp|कॉर्पोरेशन|"
    "llc|l l c|एलएलसी|"
    "llp|l l p|एलएलपी|"
    "plc"
)

for source in ["s1", "s2", "s3"]:

    con.execute(f"""
    CREATE OR REPLACE TABLE {source}_core_name_v2 AS

    SELECT
        entity_id,
        country,
        normalized_name,

        regexp_replace(
            regexp_replace(
                normalized_name,
                '(^| )({LEGAL_FORMS_V2})( |$)',
                ' ',
                'g'
            ),
            '\\s+',
            ' ',
            'g'
        ) AS core_name_v2

    FROM {source}_normalized
    """)

    con.execute(f"""
    UPDATE {source}_core_name_v2
    SET core_name_v2 = trim(core_name_v2)
    """)

print("core_name_v2 created.")

core_name_v2 created.


In [145]:
con.execute("""
SELECT
    normalized_name,
    core_name_v2
FROM s2_core_name_v2
WHERE
    normalized_name LIKE '%प्राइवेट%'
    OR normalized_name LIKE '%लिमिटेड%'
    OR normalized_name LIKE '%एलएलपी%'
    OR normalized_name LIKE '%llc%'
    OR normalized_name LIKE '%llp%'
LIMIT 50
""").df()

,normalized_name,core_name_v2
0,राम मार्केटिंग प्राइवेट लिमिटेड,राम मार्केटिंग लिमिटेड
1,आदित्य प्रॉपर्टीज एलएलपी,आदित्य प्रॉपर्टीज
2,chavira platinum chimera llc,chavira platinum chimera
3,सन कंस्ट्रक्शंस प्राइवेट लिमिटेड,सन कंस्ट्रक्शंस लिमिटेड
4,asset building coalition llc,asset building coalition
5,olszewski holding company llc llc,olszewski holding company llc
6,रियल मॉडर्न फूड लिमिटेड,रियल मॉडर्न फूड
7,keystone odyssey odyssey llc,keystone odyssey odyssey
8,core welfare society llp,core welfare society
9,cobalt llc,cobalt


In [146]:
s2_core_name_v2_recall = con.execute("""
SELECT
    COUNT(*) AS total_matches,

    COUNT(*) FILTER (
        WHERE
            s1.core_name_v2 != ''
            AND s2.core_name_v2 != ''
            AND s1.core_name_v2 = s2.core_name_v2
            AND s1.country = s2.country
    ) AS captured

FROM ground_truth_matches gt

JOIN s1_core_name_v2 s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_core_name_v2 s2
    ON gt.matched_entity_id = s2.entity_id

WHERE gt.matched_entity_id LIKE 'S2-%'
""").df()

s2_core_name_v2_recall["recall"] = (
    s2_core_name_v2_recall["captured"]
    / s2_core_name_v2_recall["total_matches"]
)

s2_core_name_v2_recall

,total_matches,captured,recall
0,3693619,1334400,0.361272


In [147]:
s3_core_name_v2_recall = con.execute("""
SELECT
    COUNT(*) AS total_matches,

    COUNT(*) FILTER (
        WHERE
            s1.core_name_v2 != ''
            AND s3.core_name_v2 != ''
            AND s1.core_name_v2 = s3.core_name_v2
            AND s1.country = s3.country
    ) AS captured

FROM ground_truth_matches gt

JOIN s1_core_name_v2 s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s3_core_name_v2 s3
    ON gt.matched_entity_id = s3.entity_id

WHERE gt.matched_entity_id LIKE 'S3-%'
""").df()

s3_core_name_v2_recall["recall"] = (
    s3_core_name_v2_recall["captured"]
    / s3_core_name_v2_recall["total_matches"]
)

s3_core_name_v2_recall

,total_matches,captured,recall
0,3944746,1368265,0.346858


In [148]:
for source in ["s1", "s2", "s3"]:

    con.execute(f"""
    CREATE OR REPLACE TABLE {source}_core_name_v2_frequency AS

    SELECT
        country,
        core_name_v2,
        COUNT(*) AS record_count

    FROM {source}_core_name_v2

    WHERE core_name_v2 != ''

    GROUP BY
        country,
        core_name_v2
    """)

print("core_name_v2 frequency tables created.")

core_name_v2 frequency tables created.


In [149]:
core_v2_volume_s2 = con.execute("""
SELECT
    COALESCE(
        SUM(
            s1.record_count * s2.record_count
        ),
        0
    )

FROM s1_core_name_v2_frequency s1

JOIN s2_core_name_v2_frequency s2

    ON s1.country = s2.country
   AND s1.core_name_v2 = s2.core_name_v2
""").fetchone()[0]

print(
    "S2 core-name-v2 candidate pairs:",
    int(core_v2_volume_s2)
)

print(
    "Average candidates/S1:",
    core_v2_volume_s2 / 2206821
)

S2 core-name-v2 candidate pairs: 30809006
Average candidates/S1: 13.96080878331319


In [150]:
core_v2_volume_s3 = con.execute("""
SELECT
    COALESCE(
        SUM(
            s1.record_count * s3.record_count
        ),
        0
    )

FROM s1_core_name_v2_frequency s1

JOIN s3_core_name_v2_frequency s3

    ON s1.country = s3.country
   AND s1.core_name_v2 = s3.core_name_v2
""").fetchone()[0]

print(
    "S3 core-name-v2 candidate pairs:",
    int(core_v2_volume_s3)
)

print(
    "Average candidates/S1:",
    core_v2_volume_s3 / 2206821
)

S3 core-name-v2 candidate pairs: 31952438
Average candidates/S1: 14.478944146353511


In [151]:
core_v2_regressions_s2 = con.execute("""
SELECT
    gt.source1_entity_id,
    gt.matched_entity_id,

    s1.normalized_name AS s1_normalized_name,
    s2.normalized_name AS s2_normalized_name,

    v1a.core_name AS s1_core_v1,
    v1b.core_name AS s2_core_v1,

    v2a.core_name_v2 AS s1_core_v2,
    v2b.core_name_v2 AS s2_core_v2

FROM ground_truth_matches gt

JOIN s1_core_name s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_core_name s2
    ON gt.matched_entity_id = s2.entity_id

JOIN s1_core_name_v2 v2a
    ON gt.source1_entity_id = v2a.entity_id

JOIN s2_core_name_v2 v2b
    ON gt.matched_entity_id = v2b.entity_id

WHERE
    gt.matched_entity_id LIKE 'S2-%'

    AND s1.core_name != ''
    AND s2.core_name != ''
    AND s1.core_name = s2.core_name

    AND NOT (
        v2a.core_name_v2 != ''
        AND v2b.core_name_v2 != ''
        AND v2a.core_name_v2 = v2b.core_name_v2
    )

LIMIT 100
""").df()

core_v2_regressions_s2

BinderException: Binder Error: Referenced table "v1a" not found!
Candidate tables: "v2a", "s1", "v2b"

LINE 9:     v1a.core_name AS s1_core_v1,
            ^

In [152]:
core_v2_regressions_s2 = con.execute("""
SELECT
    gt.source1_entity_id,
    gt.matched_entity_id,

    s1.normalized_name AS s1_normalized_name,
    s2.normalized_name AS s2_normalized_name,

    s1.core_name AS s1_core_v1,
    s2.core_name AS s2_core_v1,

    v2a.core_name_v2 AS s1_core_v2,
    v2b.core_name_v2 AS s2_core_v2

FROM ground_truth_matches gt

JOIN s1_core_name s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_core_name s2
    ON gt.matched_entity_id = s2.entity_id

JOIN s1_core_name_v2 v2a
    ON gt.source1_entity_id = v2a.entity_id

JOIN s2_core_name_v2 v2b
    ON gt.matched_entity_id = v2b.entity_id

WHERE
    gt.matched_entity_id LIKE 'S2-%'

    AND s1.core_name != ''
    AND s2.core_name != ''
    AND s1.core_name = s2.core_name

    AND NOT (
        v2a.core_name_v2 != ''
        AND v2b.core_name_v2 != ''
        AND v2a.core_name_v2 = v2b.core_name_v2
    )

LIMIT 100
""").df()

core_v2_regressions_s2

,source1_entity_id,matched_entity_id,s1_normalized_name,s2_normalized_name,s1_core_v1,s2_core_v1,s1_core_v2,s2_core_v2
0,S1-162465383,S2-952666917,arena products private limited,private arena products ltd,arena products,arena products,arena products limited,arena products
1,S1-279616287,S2-877078673,akn devcon private limited,akn devcon private,akn devcon,akn devcon,akn devcon limited,akn devcon
2,S1-9114550,S2-998041093,ko consulting private limited,ko consulting private,ko consulting,ko consulting,ko consulting limited,ko consulting
3,S1-457444169,S2-359190182,asmita polyfibres private limited,asmita polyfibres private,asmita polyfibres,asmita polyfibres,asmita polyfibres limited,asmita polyfibres
4,S1-300182868,S2-266241626,seven technology private limited,seven technology private,seven technology,seven technology,seven technology limited,seven technology
...,...,...,...,...,...,...,...,...
95,S1-218282422,S2-197004101,elon ridge private limited,elon ridge private ltd,elon ridge,elon ridge,elon ridge limited,elon ridge ltd
96,S1-108416737,S2-129284784,sree technology private limited,sree technology private ltd,sree technology,sree technology,sree technology limited,sree technology ltd
97,S1-406388261,S2-852269665,red foods private limited,red foods private,red foods,red foods,red foods limited,red foods
98,S1-819236579,S2-229952561,shree connect private limited,shree connect private ltd,shree connect,shree connect,shree connect limited,shree connect ltd


In [153]:
core_v2_regressions_s3 = con.execute("""
SELECT
    gt.source1_entity_id,
    gt.matched_entity_id,

    s1.normalized_name AS s1_normalized_name,
    s3.normalized_name AS s3_normalized_name,

    s1.core_name AS s1_core_v1,
    s3.core_name AS s3_core_v1,

    v2a.core_name_v2 AS s1_core_v2,
    v2b.core_name_v2 AS s3_core_v2

FROM ground_truth_matches gt

JOIN s1_core_name s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s3_core_name s3
    ON gt.matched_entity_id = s3.entity_id

JOIN s1_core_name_v2 v2a
    ON gt.source1_entity_id = v2a.entity_id

JOIN s3_core_name_v2 v2b
    ON gt.matched_entity_id = v2b.entity_id

WHERE
    gt.matched_entity_id LIKE 'S3-%'

    AND s1.core_name != ''
    AND s3.core_name != ''
    AND s1.core_name = s3.core_name

    AND NOT (
        v2a.core_name_v2 != ''
        AND v2b.core_name_v2 != ''
        AND v2a.core_name_v2 = v2b.core_name_v2
    )

LIMIT 100
""").df()

core_v2_regressions_s3

,source1_entity_id,matched_entity_id,s1_normalized_name,s3_normalized_name,s1_core_v1,s3_core_v1,s1_core_v2,s3_core_v2
0,S1-19984456,S3-940119092,delhi management pvt ltd,pvt delhi management limited,delhi management,delhi management,delhi management ltd,delhi management
1,S1-91409864,S3-620608562,great infratech private limited,limited great infratech private,great infratech,great infratech,great infratech limited,great infratech
2,S1-546472481,S3-15496049,vn traders private limited,vn traders private,vn traders,vn traders,vn traders limited,vn traders
3,S1-367734152,S3-760483345,industries logistics fabrication private limited,industries logistics fabrication private,industries logistics fabrication,industries logistics fabrication,industries logistics fabrication limited,industries logistics fabrication
4,S1-699763764,S3-376434533,sports india consultingprivate private limited,sports india consultingprivate private ltd,sports india consultingprivate,sports india consultingprivate,sports india consultingprivate limited,sports india consultingprivate ltd
...,...,...,...,...,...,...,...,...
95,S1-687361350,S3-751712654,green blue consultancy private limited,green blue consultancy,green blue consultancy,green blue consultancy,green blue consultancy limited,green blue consultancy
96,S1-400778820,S3-106801878,krishna traders private limited,krishna traders private,krishna traders,krishna traders,krishna traders limited,krishna traders
97,S1-824031638,S3-510935207,international ambulance pvt ltd,international ambulance,international ambulance,international ambulance,international ambulance ltd,international ambulance
98,S1-403442123,S3-846366557,dream infra private limited,dream infra private,dream infra,dream infra,dream infra limited,dream infra


In [154]:
con.execute("""
SELECT COUNT(*)
FROM ground_truth_matches gt

JOIN s1_core_name s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s2_core_name s2
    ON gt.matched_entity_id = s2.entity_id

JOIN s1_core_name_v2 v21
    ON gt.source1_entity_id = v21.entity_id

JOIN s2_core_name_v2 v22
    ON gt.matched_entity_id = v22.entity_id

WHERE gt.matched_entity_id LIKE 'S2-%'

  AND s1.core_name != ''
  AND s2.core_name != ''
  AND s1.core_name = s2.core_name

  AND NOT (
      v21.core_name_v2 != ''
      AND v22.core_name_v2 != ''
      AND v21.core_name_v2 = v22.core_name_v2
  )
""").fetchone()[0]

239259

In [155]:
con.execute("""
SELECT COUNT(*)
FROM ground_truth_matches gt

JOIN s1_core_name s1
    ON gt.source1_entity_id = s1.entity_id

JOIN s3_core_name s3
    ON gt.matched_entity_id = s3.entity_id

JOIN s1_core_name_v2 v21
    ON gt.source1_entity_id = v21.entity_id

JOIN s3_core_name_v2 v22
    ON gt.matched_entity_id = v22.entity_id

WHERE gt.matched_entity_id LIKE 'S3-%'

  AND s1.core_name != ''
  AND s3.core_name != ''
  AND s1.core_name = s3.core_name

  AND NOT (
      v21.core_name_v2 != ''
      AND v22.core_name_v2 != ''
      AND v21.core_name_v2 = v22.core_name_v2
  )
""").fetchone()[0]

252488

In [156]:
LEGAL_PATTERNS = [
    r'(^| )private( |$)',
    r'(^| )pvt( |$)',
    r'(^| )प्राइवेट( |$)',

    r'(^| )limited( |$)',
    r'(^| )ltd( |$)',
    r'(^| )लिमिटेड( |$)',

    r'(^| )incorporated( |$)',
    r'(^| )inc( |$)',

    r'(^| )corporation( |$)',
    r'(^| )corp( |$)',

    r'(^| )llc( |$)',
    r'(^| )l l c( |$)',
    r'(^| )एलएलसी( |$)',

    r'(^| )llp( |$)',
    r'(^| )l l p( |$)',
    r'(^| )एलएलपी( |$)',

    r'(^| )plc( |$)'
]

for source in ["s1", "s2", "s3"]:

    con.execute(f"""
    CREATE OR REPLACE TABLE {source}_core_name_v2_fixed AS
    SELECT
        entity_id,
        country,
        normalized_name,
        normalized_name AS core_name_v2_fixed
    FROM {source}_normalized
    """)

    for pattern in LEGAL_PATTERNS:
        con.execute(f"""
        UPDATE {source}_core_name_v2_fixed
        SET core_name_v2_fixed =
            regexp_replace(
                core_name_v2_fixed,
                '{pattern}',
                ' ',
                'g'
            )
        """)

    con.execute(f"""
    UPDATE {source}_core_name_v2_fixed
    SET core_name_v2_fixed =
        regexp_replace(
            trim(core_name_v2_fixed),
            '\\s+',
            ' ',
            'g'
        )
    """)

print("Fixed core-name v2 tables created.")

RuntimeError: Query interrupted

In [ ]:
con.execute("""
SELECT
    normalized_name,
    core_name_v2_fixed
FROM s2_core_name_v2_fixed

WHERE
    normalized_name IN (
        'arena products private limited',
        'private arena products ltd'
    )

    OR normalized_name LIKE '%प्राइवेट%'
    OR normalized_name LIKE '%लिमिटेड%'
    OR normalized_name LIKE '%एलएलपी%'

LIMIT 50
""").df()